# 재사용 이미지 VQA — Gemma 31B 2-Stage Fine-Tuning Baseline

버전: 2026-10-04 v1. 단일 모델 학습/추론용. 설정과 코드를 한 파일에 포함한다.

기본: `unsloth/gemma-4-31B-it-unsloth-bnb-4bit`, LoRA r16/alpha32,
language attention+MLP, 정답 글자 1토큰 loss, 외부→내부 각 1 epoch,
프롬프트 2회, shift0/2 확률 평균.

**작성 검증:** 셀 문법·CPU 데이터 처리·분할·보기 재매핑·확률 저장 검증.
**미실행:** 실제 모델 다운로드/GPU backward/학습/대회 이미지 추론/Public 채점.
0.97은 목표이며 이 노트북에서 실측한 성적이 아니다.

### 처음 실행

1. 01에서 설치가 필요하면 `INSTALL_PACKAGES=True`로 바꾸어 설치하고 커널 재시작.
2. 03에서 데이터 경로·정제 모드·GPU 배치를 설정.
3. `RUN_MODE="prepare"`로 CPU 데이터/분할을 확인.
4. `RUN_MODE="smoke"`로 소량 학습·추론을 확인한 후 `"full"` 실행.
5. 같은 run 재개는 `RESUME_RUN_DIR` 지정. 완성 adapter 추론은 `RUN_MODE="infer"`와 `ADAPTER_PATH` 지정.

이미지/CSV는 포함하지 않았다. `manual_csv` 모드에는 실제 정제 CSV가 필요하다.
원본 노트북은 보존했고 이전 핵심 설정도 03의 주석/프로필로 남겼다.
모든 행 수는 데이터에서 계산한다. 1,434/6,714는 고정하지 않는다.

참고 자료: 사용자 공유 글 + A186 ZIP의 `src/train_gemma4_31b.py`.
글의 hard reverse는 ZIP 최종 코드에 없어 기본 꺼짐. 교사 모델 3.5/3.8 표기는 미확인.
우리 4:1·정답 위치 균등화는 참고의 9:1·무작위 셔플과 다른 실험이다.


## 01. 환경 준비·라이브러리 설치


설치 셀은 기존 환경을 자동으로 바꾸지 않는다. 참고 ZIP의 버전 프로필과
현재 런타임을 보존하는 설치 프로필을 선택할 수 있다. 설치 후 재시작하고 다음 단계로 진행한다.
모델·데이터 인증이 필요하면 `HF_TOKEN` 환경변수에 넣는다. 토큰을 노트북에 적지 않는다.


In [ ]:
import os, sys, subprocess
INSTALL_PACKAGES = False
INSTALL_PROFILE = "reference_pins"  # reference_pins / runtime_compatible
INSTALL_OCR = False
INSTALL_PHASH = False
# GPTQModel은 기존 Qwen 환경의 버전을 설치한 뒤 해당 프로필에서 사용한다.
# 패치 필요 여부는 09의 GPTQ 옵션과 실제 backward로 확인한다.
INSTALL_GPTQMODEL = False

if INSTALL_PACKAGES:
    common_packages = ["datasets", "pandas", "numpy", "pillow", "scikit-learn", "plotly", "tqdm"]
    if INSTALL_PROFILE == "reference_pins":
        packages = ["unsloth==2026.9.7", "transformers==5.5.0", "peft==0.21.0",
                    "bitsandbytes==0.50.2", "accelerate", *common_packages]
        # 참고 ZIP: Python3.11/CUDA12.8/torch2.11.0. 실제 설치 성공 환경은 기록해야 한다.
        # 필요할 때 별도 실행:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", "torch==2.11.0",
        #     "torchvision", "--index-url", "https://download.pytorch.org/whl/cu128"])
    else:
        packages = ["unsloth", "transformers", "peft", "bitsandbytes", "accelerate", *common_packages]
    if INSTALL_OCR:
        packages.append("easyocr")
    if INSTALL_PHASH:
        packages.append("ImageHash")
    if INSTALL_GPTQMODEL:
        packages.append("gptqmodel")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
    print("설치 완료. 커널/런타임 재시작 후 INSTALL_PACKAGES=False로 되돌리세요.")
else:
    print("설치 생략. 필요하면 위 설정을 변경하세요.")


## 02. 데이터 위치·Drive·ZIP 준비


데이터를 로컬 디스크에 둔다. Drive 연결과 ZIP 해제는 선택사항이다.
압축 해제는 경로 이탈을 검사하고 암호는 입력받되 저장하지 않는다.


In [ ]:
from pathlib import Path
MOUNT_DRIVE = False
DATA_ZIP = None  # "/content/drive/MyDrive/project/data.zip"
EXTRACT_DIR = Path("/content/data") if Path("/content").exists() else Path.cwd() / "data"
ZIP_ENCRYPTED = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if DATA_ZIP:
    import zipfile, getpass
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        destination = EXTRACT_DIR.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"ZIP 경로 이탈: {member.filename}")
        pwd = getpass.getpass("ZIP 비밀번호: ").encode() if ZIP_ENCRYPTED else None
        archive.extractall(destination, pwd=pwd)
    print("압축 해제:", EXTRACT_DIR)


## 03. 라이브러리·데이터·실험 설정

이 셀에는 이전 설정 파일을 내장했다. 다음 설정 셀만 수정하면 된다. 별도 `.py` 다운로드는 필요 없다.


In [ ]:
"""재사용 VQA 베이스라인의 초기 설정. 이 셀은 공통 설정을 정의한다.

기본값은 A186 ZIP의 최종 Gemma 학습 코드를 참고한다. 분할 4:1,
학습 정답 위치 균등화, 실제 logits/확률 저장은 새 베이스 요구를 반영한다.
원본 노트북을 수정하지 않고 이전 모델의 설정을 별도 프로필로 보존한다.

python VQA_Baseline_Config.py --output VQA_Default_Config.json
python VQA_Baseline_Config.py --preset qwen27_only --hardware portable
"""

from __future__ import annotations

import argparse
from copy import deepcopy
import json
from pathlib import Path


# 새 기본값. 이전 선택지는 삭제하지 않고 아래와 PRESETS에 남긴다.
ACTIVE_PRESET = "reference_gemma31"
# ACTIVE_PRESET = "team_common"
# ACTIVE_PRESET = "qwen27_only"
# ACTIVE_PRESET = "gemma31_legacy"
# ACTIVE_PRESET = "agnes_legacy"

# ZIP 기본 배치 16 x 1, 추론 배치 32를 보존한다.
# GPU가 다르면 portable로 변경한 뒤 실제 backward/VRAM을 확인한다.
ACTIVE_HARDWARE = "reference_b200"
# ACTIVE_HARDWARE = "portable"  # batch 4 x accumulation 4, 추론 batch 4

HARDWARE_PROFILES = {
    "reference_b200": {
        "per_device_train_batch_size": 16,
        "gradient_accumulation_steps": 1,
        "per_device_eval_batch_size": 32,
        "evidence": "A186 최종 소스 기본값 및 B200 192GB 주석; 실행 로그 미제공",
    },
    "portable": {
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "per_device_eval_batch_size": 4,
        "evidence": "새 베이스용 시작값; GPU 적합성/동일 성능은 실측 필요",
    },
}

SYSTEM_PROMPT = (
    "당신은 이미지를 보고 질문에 답하는 시각 질의응답 도우미입니다. "
    "a, b, c, d 중 정확히 한 글자로만 답하고, 설명은 하지 마세요."
)
USER_INSTRUCTION = "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."

BASE = {
    "schema_version": 1,
    "artifact_kind": "configuration_only",
    "run": {"seed": 42, "run_id": None, "output_root": "runs", "mode": "finetune"},
    "data": {
        "train_csv": None, "dev_csv": None, "test_csv": None,
        "image_root": None, "sample_submission_csv": None,
        "columns": {"id": "id", "image": "path", "question": "question",
                    "choices": ["a", "b", "c", "d"], "answer": "answer"},
        "classes": ["a", "b", "c", "d"],
        "label_to_index": {"a": 0, "b": 1, "c": 2, "d": 3},
        "split": {
            "train_fraction": 0.8, "validation_fraction": 0.2,
            "method": "group_stratified", "group_key": "image_sha256",
            "optional_phash_grouping": False,
            "validation_sources": ["train_gold"],
            "dev_pseudo_labels_training_only": True,
            "reuse_manifest": True, "manifest_path": None,
            "record_realized_counts_and_ratios": True,
        },
        "cleaning": {
            "mode": "manual_csv",  # precleaned_manifest / none도 별도 실험으로 선택 가능
            "manual_csv_dir": None,
            "required_files": ["train_changes.csv", "train_delete_list.csv",
                               "dev_changes.csv", "dev_delete_list.csv"],
            "apply_option_edits": True, "apply_delete_lists": True,
            "check_before_text_matches": True,
            "answer_letter_edit": False,
            "require_cleaning_provenance": True,
            "teacher_top2_review": {
                "enabled": False, "model_id": None,
                "action": "flag_for_human_review_only",
                "checkpoint_status": "글 Qwen3.8 / ZIP README Qwen3.5; 정확한 교사 미확인",
            },
        },
        "dev_votes": {
            "enabled": True, "columns_pattern": "answer[0-9]+",
            "keep_patterns": ["3", "3:1", "3:1:1"],
            "numeric_to_letter": {"1": "a", "2": "b", "3": "c", "4": "d"},
            "target": "majority", "source_tag_required": True,
        },
        "leakage": {
            "exclude_validation_from_all_training_sources": True,
            "check_ids": True, "check_image_paths": True,
            "check_content_hash_across_sources": True,
            "check_test_overlap_and_record": True,
        },
        "training_option_order": {
            "mode": "balanced_target_positions",  # original / random_shuffle도 선택 가능
            "target_ratios": [0.25, 0.25, 0.25, 0.25],
            "deterministic_seeded": True, "remap_answer_with_options": True,
            "reject_unsafe_position_references": True,
            "report_exempt_samples_and_realized_balance": True,
            "validation_original_order": True, "test_original_order": True,
        },
        "hard_reverse": {
            "enabled": False, "scope": "training_only",
            "order": [3, 2, 1, 0], "answer_index_map": [3, 2, 1, 0],
            "evidence": "A186 실험 코드 기능; 최종 스크립트에는 없음",
        },
    },
    "image": {
        "exif_transpose": True, "color_mode": "RGB",
        "resize_mode": "native_aspect",  # letterbox 선택 시 아래 설정 사용
        "longest_side_cap": 2048, "max_soft_tokens": 1120,
        "letterbox": {"side": 1024, "fill_rgb": [255, 255, 255], "enabled": False},
        "augmentation": {
            "external": ["orig"],
            "target": ["orig", "compression", "blur", "affine"],
            "jpeg_quality": 70, "blur_radius": 1.0,
            "affine_translate_fraction": 0.03, "affine_shear_degrees": 5.0,
            "fill_rgb": [255, 255, 255],
            "validation": ["orig"], "test": ["orig"],
            "legacy_photometric": {"enabled": False,
                "source": "hybrid 노트북의 밝기/대비/색상/노이즈/JPEG 옵션 보존 대상"},
        },
    },
    "extra_context": {
        "mode": "none",  # none / ocr / dino / ocr_dino
        "ocr": {"backend": "easyocr", "languages": ["ko", "en"],
                "cache_confidence": 0.25, "prompt_confidence": 0.40, "max_items": 15},
        "dino": {"model_id": "IDEA-Research/grounding-dino-tiny",
                 "threshold": 0.30, "max_items": 12, "object_queries": None},
        "cache_key": "image_hash+preprocess_config+tool_revision",
        "preserve_question_and_image_priority": True,
        "compare_on_same_validation_manifest": True,
    },
    "prompt": {
        "system": SYSTEM_PROMPT, "instruction": USER_INSTRUCTION,
        "repeat": 2, "repeat_separator": "\n\n",
        "repeat_content": "question+four_choices+instruction",
        "image_occurrences": 1,
        "template_kwargs": {"enable_thinking": False},
        "answer_prefix": "<|channel>thought\n<channel|>",
        "append_prefix_only_if_missing": True,
        "assert_each_answer_is_one_token": True,
        "verify_next_token_answer_boundary": True,
    },
    "model": {
        "model_id": "unsloth/gemma-4-31B-it-unsloth-bnb-4bit",
        "revision": None, "record_resolved_revision": True,
        "backend": "unsloth_fastvision", "quantization": "bitsandbytes_4bit",
        "load_in_4bit": True, "compute_dtype": "bfloat16",
        "max_seq_length": 4096, "gradient_checkpointing": True,
        "trust_remote_code": False,
        "environment_before_import": {
            "UNSLOTH_DISABLE_STATISTICS": "1", "UNSLOTH_RETURN_LOGITS": "1",
            "TOKENIZERS_PARALLELISM": "false",
        },
    },
    "lora": {
        "r": 16, "alpha": 32, "dropout": 0.0, "bias": "none",
        "finetune_vision_layers": False, "finetune_language_layers": True,
        "finetune_attention_modules": True, "finetune_mlp_modules": True,
        "record_resolved_target_module_names": True,
    },
    "loss": {
        "mode": "next_answer_token_only", "supervised_tokens_per_sample": 1,
        "normalization": "full_vocabulary_cross_entropy",
        "exclude_prompt_image_reasoning_eos_padding_from_loss": True,
        "padding_side": "left",
    },
    "external_data": {
        "enabled": True,
        "mtvqa": {"repo_id": "ByteDance/MTVQA", "splits": ["train", "test"],
                  "languages": ["KO", "KR"], "max_answer_chars": 40,
                  "distractor_pool": 50, "convert_to_four_choices": True,
                  "distractor_priority": ["other_answers_same_image", "similar_type_and_length"],
                  "exclude_duplicate_and_gold_substring_choices": True},
        "kreta": {"repo_id": "tabtoyou/KRETA", "split": "test",
                  "choice_columns": ["A", "B", "C", "D"], "answer_to_lowercase": True},
        "dataset_revision": None, "record_resolved_revisions": True,
        "record_external_split_used_for_training": True,
    },
    "training": {
        "optimizer": "adamw_8bit", "scheduler": "cosine",
        "warmup_ratio": 0.05, "weight_decay": 0.01, "max_grad_norm": 1.0,
        "logging_steps": 10, "save_steps": 200, "save_total_limit": 2,
        "num_workers": None, "report_to": "none",
        "preflight_backward_required": True, "benchmark_steps": 20,
        "stages": [
            {"name": "external", "epochs": 1.0, "learning_rate": 1e-4,
             "augmentation": ["orig"], "save_adapter": "adapter_external"},
            {"name": "target", "epochs": 1.0, "learning_rate": 1e-4,
             "augmentation": ["orig", "compression", "blur", "affine"],
             "save_adapter": "adapter"},
        ],
        "carry_same_adapter_external_to_target": True,
        "new_optimizer_and_scheduler_per_stage": True,
        "checkpoint_selection": "last_stage_final",
        "resume_with_config_and_data_hash_checks": True,
    },
    "inference": {
        "mode": "direct_choice_logits", "temperature": 1.0,
        "choice_token_order": ["a", "b", "c", "d"],
        "shifts": [0, 2], "map_back_to_original_order": True,
        "aggregation": "arithmetic_probability_mean",
        "generation_diagnostic": {"enabled": True, "sample_count": 8},
        "validation_shift_calibration": {
            "enabled": True, "candidate_shifts": [0, 1, 2, 3],
            "apply_selected_shift_instead_of_tta": False,
            "selection_policy": "record_reference_recipe_without_automatic_search",
        },
        "highres_1024_tta4": {"enabled": False, "side": 1024,
                              "shifts": [0, 1, 2, 3], "resize_mode": "letterbox"},
        "same_model_routing": {"enabled": False, "confidence_threshold": None,
                               "margin_threshold": None},
        "save_partial_predictions": True, "resume_only_matching_run_signature": True,
    },
    "artifacts": {
        "class_order": ["a", "b", "c", "d"], "expected_counts": "derive_from_manifests",
        "required": ["probs/valid_logits.npy", "probs/valid_probs.npy",
                     "probs/valid_labels.npy", "probs/valid_ids.json",
                     "probs/test_logits.npy", "probs/test_probs.npy", "probs/test_ids.json",
                     "validation_predictions.csv", "validation_summary.json",
                     "analysis/shift_calibration.csv", "analysis/selected_shift.json"],
        "additional_tta": ["probs/valid_shift_logits.npy", "probs/valid_shift_probs.npy",
                           "probs/test_shift_logits.npy", "probs/test_shift_probs.npy",
                           "probs/valid_reference_probs.npy", "probs/test_reference_probs.npy"],
        "metadata_file": "probs/probability_metadata.json",
        "logits_semantics": "actual_restored_logits_of_reference_shift",
        "logits_reference_shift": 0,
        "probs_semantics": "final_arithmetic_mean_of_restored_shift_probabilities",
        "raw_logits_are_not_log_of_averaged_probs": True,
        "id_order": "validation_manifest_or_original_test_order",
        "submission_filename": "submission_{run_id}.csv",
        "submission_columns": ["id", "answer"],
        "charts": ["loss_lr_vram", "before_after_accuracy", "class_accuracy_confusion",
                   "shift_tta_comparison", "source_and_question_type_accuracy"],
        "run_comparison_requires_same_validation_ids_and_label_hash": True,
    },
    "environment_reference": {
        "source": "A186 ZIP requirements.txt; 실제 환경 로그/호환성 미확인",
        "python": "3.11", "cuda": "12.8", "torch": "2.11.0",
        "unsloth": "2026.9.7", "transformers": "5.5.0",
        "peft": "0.21.0", "bitsandbytes": "0.50.2",
        "record_actual_versions_and_freeze": True,
    },
    "evidence": {
        "reference_score": {"public": 0.97229,
                            "status": "solution_report_only_no_submission_or_score_log"},
        "desired_public": 0.97, "score_guaranteed": False,
        "reference_recipe_source": "A186_민구없는_민구팀.zip/src/train_gemma4_31b.py",
        "execution_status": "not_trained_not_inferred_config_prepared",
    },
}

# 현재 노트북에 있는 핵심 설정 보존. 과거 성적을 낸 설정으로 확정하지 않는다.
# 모델을 바꾸어도 새 4:1 분할·NPY 규격은 공통이다. 완전한 과거 재현에는
# 원본 노트북의 backend, manifest, processor, template, 실행 로그가 더 필요하다.
PRESETS = {
    "reference_gemma31": {"kind": "reference_recipe_for_new_baseline"},
    "team_common": {"model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
                    "backend": "transformers_peft", "quantization": "none",
                    "r": 16, "alpha": 32, "lr": 2e-4, "max_seq_length": 4096,
                    "source": "팀_공통_베이스라인.ipynb"},
    "qwen27_only": {"model_id": "btbtyler09/Qwen3.8-27B-GPTQ-4bit",
                    "backend": "gptq_peft_with_backward_check", "quantization": "gptq_4bit",
                    "r": 8, "alpha": 16, "lr": 5e-5, "max_seq_length": 4096,
                    "source": "Qwen38_27B_clean7169_AgnesStyle_A100.ipynb"},
    "gemma31_legacy": {"model_id": "google/gemma-4-31B-it",
                       "backend": "transformers_peft", "quantization": "none",
                       "r": 8, "alpha": 16, "lr": 5e-5, "max_seq_length": 2048,
                       "source": "gemma4_31b_TRAINING(1).ipynb"},
    "agnes_legacy": {"model_id": "Agnes-AI/Agnes-3.0-Flash",
                     "backend": "transformers_peft_delta_targets", "quantization": "none",
                     "r": 8, "alpha": 16, "lr": 5e-5, "max_seq_length": 2048,
                     "source": "Agnes-3.0-Flash_clean7169_full_v3(2)(1).ipynb"},
}


def build_config(preset=ACTIVE_PRESET, hardware=ACTIVE_HARDWARE, extra_context="none"):
    """독립된 설정을 반환한다. 모델/데이터/GPU를 로드하지 않는다."""
    cfg = deepcopy(BASE)
    old = PRESETS[preset]
    cfg["preset"] = preset
    cfg["hardware_profile"] = hardware
    cfg["training"].update(deepcopy(HARDWARE_PROFILES[hardware]))
    cfg["extra_context"]["mode"] = extra_context
    if preset != "reference_gemma31":
        cfg["hardware_profile"] = "legacy_single_gpu_starting_values"
        cfg["model"].update(model_id=old["model_id"], backend=old["backend"],
                            quantization=old["quantization"],
                            load_in_4bit=old["quantization"] != "none",
                            max_seq_length=old["max_seq_length"],
                            environment_before_import={})
        cfg["lora"].update(r=old["r"], alpha=old["alpha"], dropout=0.05,
                           finetune_mlp_modules=False)
        cfg["lora"]["resolve_backend_specific_targets_from_source"] = old["source"]
        cfg["external_data"]["enabled"] = False
        cfg["prompt"].update(repeat=1, answer_prefix=None, template_kwargs={})
        cfg["image"].update(max_soft_tokens=None, longest_side_cap=None)
        cfg["image"]["augmentation"]["target"] = ["orig"]
        cfg["data"]["cleaning"].update(mode="precleaned_manifest", required_files=[])
        cfg["data"]["dev_votes"]["enabled"] = False
        cfg["training"].update(
            stages=[{"name": "target", "epochs": 1.0, "learning_rate": old["lr"],
                     "augmentation": ["orig"], "save_adapter": "adapter"}],
            carry_same_adapter_external_to_target=False,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            per_device_eval_batch_size=1,
            evidence="기존 노트북의 batch1/accumulation8 시작값; reference 하드웨어 배치는 적용 안 함",
        )
        cfg["inference"]["shifts"] = [0]
        cfg["environment_reference"] = {
            "source": "선택한 기존 노트북별 환경 확인 필요", "record_actual_versions_and_freeze": True
        }
        cfg["evidence"].update(
            previous_notebook=old["source"],
            compatibility_note="핵심 설정 보존 프로필; 원본 실행을 그대로 재현하는 실행기는 아님",
        )
    validate_config(cfg)
    return cfg


def validate_config(cfg):
    """초기 설정의 내부 일관성만 검사한다. 입력 파일/실행 환경 확인은 별도로 필요하다."""
    split = cfg["data"]["split"]
    if abs(split["train_fraction"] + split["validation_fraction"] - 1.0) > 1e-9:
        raise ValueError("train/validation 분할 비율 합은 1이어야 합니다")
    if cfg["artifacts"]["class_order"] != cfg["data"]["classes"]:
        raise ValueError("데이터와 저장 확률의 클래스 순서가 다릅니다")
    shifts = cfg["inference"]["shifts"]
    if not shifts or len(set(shifts)) != len(shifts) or not set(shifts) <= {0, 1, 2, 3}:
        raise ValueError("shift는 0~3 범위의 중복 없는 목록이어야 합니다")
    if cfg["artifacts"]["logits_reference_shift"] not in shifts:
        raise ValueError("실제 logits 저장용 기준 shift가 추론 목록에 있어야 합니다")
    if cfg["loss"]["supervised_tokens_per_sample"] != 1:
        raise ValueError("현재 기본 loss는 정답 글자 1토큰을 감독합니다")
    stages = cfg["training"]["stages"]
    if cfg["external_data"]["enabled"] and [s["name"] for s in stages] != ["external", "target"]:
        raise ValueError("외부→내부 2단계의 순서가 다릅니다")


In [ ]:
# -------- 수정할 기본 실행 설정 --------
AUTO_RUN = True                 # False: 함수/설정만 읽고 정의, 실제 실행 생략
RUN_MODE = "prepare"            # prepare / smoke / full / infer
PRESET = "reference_gemma31"
# PRESET = "team_common"
# PRESET = "qwen27_only"
# PRESET = "gemma31_legacy"
# PRESET = "agnes_legacy"
HARDWARE = "reference_b200"
# HARDWARE = "portable"          # batch4 x accum4; GPU별 backward 점검 필요
AUX_MODE = "none"               # none / ocr / dino / ocr_dino

CFG = build_config(PRESET, HARDWARE, AUX_MODE)
CFG["artifact_kind"] = "notebook_pipeline_v1"
CFG["data"]["train_csv"] = None  # "/content/data/train.csv" 또는 정제본 CSV
CFG["data"]["dev_csv"] = None    # 없으면 None; pseudo-label은 학습에만 추가
CFG["data"]["test_csv"] = None   # "/content/data/test.csv"
CFG["data"]["sample_submission_csv"] = None
CFG["data"]["image_root"] = str(EXTRACT_DIR)
CFG["data"]["provenance"] = {"column":None,"path_prefix_map":{},
    "gold_values":["train_gold"],"declare_combined_all_gold":False}
# 통합 정제 CSV에서 path가 train/... 또는 dev/...이면 명시적으로 사용:
# CFG["data"]["provenance"]["path_prefix_map"]={"train":"train_gold","dev":"dev_pseudo"}
# 또는 CSV 출처 열을 지정하고 gold_values도 해당 값으로 설정:
# CFG["data"]["provenance"]["column"]="label_source"
# CFG["data"]["provenance"]["gold_values"]=["train_original"]
CFG["data"]["cleaning"]["mode"] = "manual_csv"
# CFG["data"]["cleaning"]["mode"] = "precleaned_manifest"
# CFG["data"]["cleaning"]["mode"] = "none"  # 별도 정제 없는 비교 실험
CFG["data"]["cleaning"]["manual_csv_dir"] = None
CLEANING_PROVENANCE = None        # 정제본 manifest/audit JSON/CSV 경로
RESUME_RUN_DIR = None             # 기존 run 전체 재개용 경로
ADAPTER_PATH = None               # infer에서 사용할 완성 adapter
CFG["run"]["output_root"] = "/content/training_runs" if Path("/content").exists() else "training_runs"
SPLIT_MANIFEST = None             # 지정하면 같은 데이터의 분할을 재사용
SMOKE_BASE_ROWS = 16
SMOKE_VALID_ROWS = 8
SMOKE_TEST_ROWS = 8
SMOKE_MAX_STEPS = 2
SMOKE_USE_EXTERNAL = False       # 첫 smoke는 외부 다운로드 생략; full은 CFG대로 Stage1 실행
RUN_BENCHMARK = False
RUN_HIGHRES_TTA4 = False
COMPARE_RUN_DIRS = []             # ["/path/to/run1", "/path/to/run2"]
ENABLE_GPTQ_TRITON_LEGACY_PATCH = False
# 예전 Qwen Triton 8-return 패치는 해당 환경에서만 켠다.
CFG["model"]["trust_remote_code"] = PRESET in {"qwen27_only", "agnes_legacy"}
CFG["image"]["augmentation"]["legacy_photometric"].update(
    enabled=False, brightness=0.12, contrast=0.12, probability=0.35,
    hue_saturation_probability=0.20, noise_probability=0.15,
    jpeg_probability=0.20, jpeg_quality_min=82, jpeg_quality_max=100)
CFG["extra_context"]["dino"]["object_queries"] = "person. bottle. cup. can. paper. box. bag. sign. vehicle. food."
# 위 DINO queries는 범용 시작값이다. 기존 hybrid의 정확한 객체 목록을 재현한다고 주장하지 않는다.
CFG["extra_context"]["device"] = "cpu"
CFG["extra_context"]["dino"]["revision"] = None
CFG["training"]["num_workers"] = 0  # PIL/캐시/전역 processor를 쓰는 v1은 안정적인 단일 worker
CFG["training"]["checkpoint_selection"] = "last_stage_final"
# CFG["training"]["checkpoint_selection"] = "best_retained_checkpoint"
CFG["inference"]["same_model_routing"].update(enabled=False, confidence_threshold=0.80,
    margin_threshold=0.20, extra_shifts=[1,3])

RUNTIME = {"mode": RUN_MODE, "smoke_base_rows": SMOKE_BASE_ROWS,
           "smoke_valid_rows": SMOKE_VALID_ROWS, "smoke_test_rows": SMOKE_TEST_ROWS,
           "smoke_max_steps": SMOKE_MAX_STEPS, "cleaning_provenance": CLEANING_PROVENANCE,
           "adapter_path": ADAPTER_PATH, "gptq_patch": ENABLE_GPTQ_TRITON_LEGACY_PATCH,
           "highres_tta4": RUN_HIGHRES_TTA4,"smoke_use_external":SMOKE_USE_EXTERNAL}
assert RUN_MODE in {"prepare", "smoke", "full", "infer"}
GPU_RUN = AUTO_RUN and RUN_MODE in {"smoke", "full", "infer"}
os.environ.update(CFG["model"]["environment_before_import"])
if GPU_RUN and CFG["model"]["backend"] == "unsloth_fastvision":
    import unsloth  # transformers/torch보다 먼저 import
validate_config(CFG)
print("모델:", CFG["model"]["model_id"], "| mode:", RUN_MODE, "| aux:", AUX_MODE)
print("LoRA:", CFG["lora"]["r"], CFG["lora"]["alpha"], "| shifts:", CFG["inference"]["shifts"])


## 04. CSV·이미지·정제 기록 검증


열 이름이 다르면 03의 `columns`를 바꾼다. 이미지 연결·중복 ID·빈 보기·잘못된 답을 검사한다.
`manual_csv`는 실제 CSV를 요구한다. `precleaned_manifest`는 정제 출처 파일을 요구한다.
Top-2 검토 CSV 기능은 24에 있으며 자동 정답 수정/삭제로 사용하지 않는다.


In [ ]:
import hashlib, json, math, io, random, re, time, gc, inspect, platform, warnings
from datetime import datetime, timezone
from collections import Counter
from copy import deepcopy
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter, ImageEnhance

CHOICES = CFG["data"]["classes"]
SEED = CFG["run"]["seed"]
random.seed(SEED)
np.random.seed(SEED)

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(2**20), b""):
            h.update(chunk)
    return h.hexdigest()

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                      default=str).encode()).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)

def save_npy(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path.with_name(path.name + ".tmp"), "wb") as f:
        np.save(f, array, allow_pickle=False)
    path.with_name(path.name + ".tmp").replace(path)

def resolve_image(rel, csv_path, root):
    for parent in [Path(csv_path).parent, Path(root)]:
        candidate = (parent / str(rel)).expanduser()
        if candidate.is_file():
            return str(candidate.resolve())
    raise FileNotFoundError(f"이미지 없음: {rel} | CSV={csv_path} | root={root}")

def normalize_csv(path, labeled, source):
    if not path:
        raise ValueError(f"03에서 {source} CSV 경로를 설정하세요.")
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    column = CFG["data"]["columns"]
    rename = {column["id"]: "id", column["image"]: "path", column["question"]: "question"}
    rename.update(dict(zip(column["choices"], CHOICES)))
    if labeled:
        rename[column["answer"]] = "answer"
    frame = frame.rename(columns=rename)
    required = ["id", "path", "question", *CHOICES] + (["answer"] if labeled else [])
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{source} 필수 열 없음: {sorted(missing)}")
    if frame["id"].eq("").any() or frame["id"].duplicated().any():
        raise ValueError(f"{source} 빈/중복 ID")
    if frame[["question", *CHOICES]].apply(lambda s: s.str.strip().eq("")).any().any():
        raise ValueError(f"{source} 빈 질문/보기")
    if labeled:
        frame["answer"] = frame["answer"].str.strip().str.lower()
        if not frame["answer"].isin(CHOICES).all():
            raise ValueError(f"{source} 잘못된 answer; a/b/c/d 매핑을 확인하세요.")
    frame["_img"] = [resolve_image(p, path, CFG["data"]["image_root"]) for p in frame["path"]]
    hashes = {}
    for image in dict.fromkeys(frame["_img"]):
        with Image.open(image) as img:
            img.verify()
        hashes[image] = hash_file(image)
    frame["_hash"] = frame["_img"].map(hashes)
    provenance=CFG["data"].get("provenance",{})
    source_column=provenance.get("column") if source=="train_gold" else None
    prefixes=provenance.get("path_prefix_map",{}) if source=="train_gold" else {}
    if source_column and prefixes:
        raise ValueError("출처 열과 path prefix mapping 중 하나를 선택하세요.")
    if source_column:
        if source_column not in frame:
            raise ValueError(f"정제 출처 열 없음: {source_column}")
        frame["_src"]=frame[source_column]
    elif prefixes:
        raw_prefix=frame.path.map(lambda p:Path(p).parts[0])
        if not raw_prefix.isin(prefixes).all():
            raise ValueError("통합 CSV 이미지 path prefix를 모두 출처 mapping에 넣으세요.")
        frame["_src"]=raw_prefix.map(prefixes)
    else:
        if source=="train_gold" and "combined" in Path(path).stem.lower() and not provenance.get("declare_combined_all_gold"):
            raise ValueError("통합 CSV는 provenance 열/path prefix를 설정하세요. 모든 답이 실제 골드일 때만 declare_combined_all_gold=True.")
        frame["_src"] = source
    frame["_is_gold"]=frame._src.isin(provenance.get("gold_values",["train_gold"]))
    return frame

def key(v):
    return Path(str(v)).stem.lower()

def apply_manual_cleaning(frame, source, audit):
    policy = CFG["data"]["cleaning"]
    if policy["mode"] == "none":
        return frame.copy()
    if policy["mode"] == "precleaned_manifest":
        if not CLEANING_PROVENANCE or not Path(CLEANING_PROVENANCE).is_file():
            raise FileNotFoundError("정제본 모드: CLEANING_PROVENANCE에 실제 manifest/audit 경로를 넣으세요.")
        return frame.copy()
    if policy["mode"] != "manual_csv":
        raise ValueError("지원 정제 모드: manual_csv / precleaned_manifest / none")
    csv_path = CFG["data"]["train_csv"] if source == "train" else CFG["data"]["dev_csv"] or CFG["data"]["train_csv"]
    folder = Path(policy["manual_csv_dir"] or Path(csv_path).parent)
    paths = [folder / f"{source}_changes.csv", folder / f"{source}_delete_list.csv"]
    if not all(p.is_file() for p in paths):
        raise FileNotFoundError("실제 정제 CSV 필요: " + ", ".join(map(str, paths)))
    changes, deletes = [pd.read_csv(p, dtype=str, keep_default_na=False) for p in paths]
    if not {"id", "action"}.issubset(changes) or "id" not in deletes:
        raise ValueError("changes.csv: id/action/field/before/after, delete_list.csv: id 필요")
    out = frame.copy()
    for row in changes.loc[changes["action"].eq("edit_option")].to_dict("records"):
        field = row["field"].strip().lower()
        if field not in CHOICES:
            raise ValueError(f"edit_option 대상 오류: {field}")
        mask = out["id"].map(key).eq(key(row["id"]))
        if mask.sum() != 1:
            raise ValueError(f"정제 ID 대응 오류: {row['id']} ({mask.sum()} rows)")
        before = str(out.loc[mask, field].iloc[0]).strip()
        if before != str(row["before"]).strip():
            try:
                equal = float(before) == float(row["before"])
            except (ValueError, TypeError):
                equal = False
            if not equal:
                raise ValueError(f"정제 before 불일치: {row['id']} {field}")
        out.loc[mask, field] = row["after"]
        audit.append({"source": source, "id": row["id"], "action": "edit_option",
                      "field": field, "before": before, "after": row["after"]})
    ignored = changes.loc[~changes["action"].isin(["delete", "edit_option"])]
    if len(ignored):
        warnings.warn(f"{source}: 최종 참고 코드 밖의 작업 {Counter(ignored.action)}; 적용 안 함")
    remove = set(deletes.id.map(key)) | set(changes.loc[changes.action.eq("delete"), "id"].map(key))
    mask = out.id.map(key).isin(remove)
    audit.extend({"source": source, "id": rid, "action": "delete"} for rid in out.loc[mask, "id"])
    return out.loc[~mask].reset_index(drop=True)

def run_context():
    root = Path(CFG["run"]["output_root"])
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-" + PRESET + "-" + RUN_MODE
    out = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else root / run_id
    out.mkdir(parents=True, exist_ok=True)
    for child in ["analysis", "probs", "logs", "checkpoints", "charts"]:
        (out / child).mkdir(exist_ok=True)
    signature = digest({"config": CFG, "runtime": RUNTIME})
    config_path = out / "config.json"
    if config_path.exists():
        if json.loads(config_path.read_text())["signature"] != signature:
            raise ValueError("재개 run의 설정이 다릅니다. 새 run으로 실행하세요.")
    else:
        save_json(config_path, {"signature": signature, "config": CFG, "runtime": RUNTIME})
    return out, signature

if AUTO_RUN:
    RUN_DIR, CONFIG_SIGNATURE = run_context()
    clean_audit = []
    primary_all = normalize_csv(CFG["data"]["train_csv"], True, "train_gold")
    gold_all = apply_manual_cleaning(primary_all.loc[primary_all._is_gold], "train", clean_audit)
    combined_pseudo = primary_all.loc[~primary_all._is_gold].copy()
    if len(combined_pseudo):
        combined_pseudo = apply_manual_cleaning(combined_pseudo, "dev", clean_audit)
    if not len(gold_all):
        raise ValueError("validation용 골드 데이터가 없습니다. 출처 mapping을 확인하세요.")
    print("정제 후 골드 train:", len(gold_all), "| 이미지:", gold_all._hash.nunique(),"| 통합 pseudo:",len(combined_pseudo))


## 05. 이미지 그룹 4:1 분할·validation 고정


5-fold StratifiedGroupKFold 후보 중 크기/정답 분포 편차가 작은 fold를 선택한다.
동일 SHA 이미지 그룹은 나누지 않는다. pHash 동일값 그룹 결합도 선택 가능하다.
분할 manifest는 이미지·ID·정답 hash로 데이터 변경을 검사한다. 그룹 제약의 오차는 보고한다.


In [ ]:
def add_group_keys(frame):
    out = frame.copy()
    out["_group"] = out["_hash"]
    if CFG["data"]["split"]["optional_phash_grouping"]:
        import imagehash
        parent = {h: h for h in out._hash.unique()}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        seen = {}
        for p, h in out[["_img", "_hash"]].drop_duplicates().itertuples(index=False, name=None):
            with Image.open(p) as im:
                ph = str(imagehash.phash(ImageOps.exif_transpose(im).convert("RGB")))
            if ph in seen:
                parent[find(h)] = find(seen[ph])
            seen[ph] = h
        out["_group"] = out._hash.map(find)
    return out

def split_gold(frame, manifest_path):
    from sklearn.model_selection import StratifiedGroupKFold
    frame = add_group_keys(frame).reset_index(drop=True)
    signature = digest(frame[["id", "answer", "_hash", "_group", "question", *CHOICES]].to_dict("records"))
    path = Path(manifest_path)
    if path.exists():
        saved = json.loads(path.read_text())
        if saved["data_signature"] != signature:
            raise ValueError("분할 manifest와 데이터가 다릅니다. 새 manifest를 사용하세요.")
        by_id = frame.set_index("id", drop=False)
        train = by_id.loc[saved["train_ids"]].reset_index(drop=True)
        valid = by_id.loc[saved["valid_ids"]].reset_index(drop=True)
    else:
        fraction = CFG["data"]["split"]["validation_fraction"]
        if abs(fraction - 0.2) > 1e-9:
            raise ValueError("v1 grouped split은 4:1을 구현합니다. 다른 비율은 분할 구현을 바꾸세요.")
        if frame._group.nunique() < 5:
            raise ValueError("4:1 grouped split에 최소 5개 이미지 그룹이 필요합니다.")
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        full = frame.answer.value_counts(normalize=True).reindex(CHOICES, fill_value=0).to_numpy()
        candidates = []
        for tr, va in splitter.split(frame, frame.answer, frame._group):
            dist = frame.iloc[va].answer.value_counts(normalize=True).reindex(CHOICES, fill_value=0).to_numpy()
            score = abs(len(va)/len(frame)-fraction) + np.abs(dist-full).mean()
            candidates.append((float(score), tr, va))
        _, tr, va = min(candidates, key=lambda t: t[0])
        train, valid = frame.iloc[tr].copy(), frame.iloc[va].copy()
        saved = {"data_signature": signature, "seed": SEED, "requested_train_fraction": 0.8,
                 "method": "StratifiedGroupKFold5_best_distribution_fold",
                 "train_ids": train.id.tolist(), "valid_ids": valid.id.tolist()}
        save_json(path, saved)
    if set(train._group) & set(valid._group):
        raise AssertionError("학습/validation 이미지 그룹 누수")
    if set(train.id) & set(valid.id):
        raise AssertionError("학습/validation ID 누수")
    return train.reset_index(drop=True), valid.reset_index(drop=True), saved

def remove_leakage(frame, valid, name):
    ids = set(valid.id.map(key))
    hashes = set(valid._hash)
    paths = set(valid._img)
    mask = frame.id.map(key).isin(ids) | frame._hash.isin(hashes) | frame._img.isin(paths)
    removed = frame.loc[mask, ["id", "_src", "_hash"]].copy()
    if len(removed) and AUTO_RUN:
        removed.to_csv(RUN_DIR / "analysis" / f"leakage_removed_{name}.csv", index=False)
    return frame.loc[~mask].reset_index(drop=True)

def vote_dev(frame):
    columns = sorted([c for c in frame if re.fullmatch(r"answer\d+", c)], key=lambda c: int(c[6:]))
    if not columns:
        raise ValueError("dev 투표 패턴을 만들 answer1~N 열이 없습니다.")
    def votes(row):
        ans = []
        for c in columns:
            value = str(row[c]).strip().lower()
            if value in {"", "nan", "none"}:
                continue
            value = {"1":"a","2":"b","3":"c","4":"d", "1.0":"a","2.0":"b","3.0":"c","4.0":"d"}.get(value, value)
            if value not in CHOICES:
                raise ValueError(f"dev 응답 해석 오류: {row['id']} {value}")
            ans.append(value)
        counter = Counter(ans)
        pattern = ":".join(map(str, sorted(counter.values(), reverse=True)))
        return pattern, max(CHOICES, key=lambda c: counter[c]) if counter else None
    out = frame.copy()
    values = [votes(r) for r in out.to_dict("records")]
    out["_vote_pattern"] = [v[0] for v in values]
    out["answer"] = [v[1] for v in values]
    return out.loc[out._vote_pattern.isin(CFG["data"]["dev_votes"]["keep_patterns"])].reset_index(drop=True)

if AUTO_RUN:
    manifest_path = SPLIT_MANIFEST or (RUN_DIR.parent / "splits" / (digest(gold_all[["id","answer","_hash"]].to_dict("records"))[:16] + "-80_20.json"))
    train_gold, valid_full, split_manifest = split_gold(gold_all, manifest_path)
    save_json(RUN_DIR / "split_manifest.json", split_manifest)
    parts = [train_gold]
    if len(combined_pseudo):
        parts.append(remove_leakage(combined_pseudo, valid_full, "combined_pseudo"))
    if CFG["data"]["dev_csv"]:
        dev = normalize_csv(CFG["data"]["dev_csv"], False, "dev_pseudo")
        dev = apply_manual_cleaning(dev, "dev", clean_audit)
        dev = vote_dev(dev)
        dev = remove_leakage(dev, valid_full, "dev")
        if set(train_gold.id) & set(dev.id):
            raise ValueError("train/dev ID 중복. 출처 ID 연결을 확인하세요.")
        parts.append(dev)
    target_full = pd.concat(parts, ignore_index=True)
    if target_full.id.duplicated().any():
        raise ValueError("합쳐진 학습 데이터 ID 중복. 출처 및 중복 추가를 확인하세요.")
    target_full.to_csv(RUN_DIR / "train_base_rows.csv", index=False, encoding="utf-8-sig")
    valid_full.to_csv(RUN_DIR / "validation_rows.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(clean_audit).to_csv(RUN_DIR / "analysis" / "cleaning_audit.csv", index=False)
    if CFG["data"]["test_csv"]:
        test_full = normalize_csv(CFG["data"]["test_csv"], False, "test")
        overlap = target_full._hash.isin(set(test_full._hash))
        target_full.loc[overlap, ["id","_src","_hash"]].to_csv(RUN_DIR / "analysis" / "train_test_overlap.csv", index=False)
    else:
        test_full = None
    valid_df = valid_full.head(SMOKE_VALID_ROWS).copy() if RUN_MODE == "smoke" else valid_full
    target_df = target_full.head(SMOKE_BASE_ROWS).copy() if RUN_MODE == "smoke" else target_full
    test_df = test_full.head(SMOKE_TEST_ROWS).copy() if RUN_MODE == "smoke" and test_full is not None else test_full
    data_fingerprint = digest({"train": target_full[["id","_hash","_src","answer","question",*CHOICES]].to_dict("records"),
        "valid":valid_full[["id","_hash","_src","answer","question",*CHOICES]].to_dict("records"),
        "test":None if test_full is None else test_full[["id","_hash","question",*CHOICES]].to_dict("records"),
        "cleaning_provenance": hash_file(CLEANING_PROVENANCE) if CLEANING_PROVENANCE else None})
    signature_path = RUN_DIR / "data_signature.json"
    if signature_path.exists() and json.loads(signature_path.read_text())["signature"] != data_fingerprint:
        raise ValueError("재개 run의 데이터가 다릅니다.")
    save_json(signature_path, {"signature": data_fingerprint})
    report = {"gold_train": len(train_gold), "gold_valid": len(valid_full), "train_with_dev": len(target_full),
        "realized_train_fraction": len(train_gold)/len(gold_all),
        "validation_class_counts": valid_full.answer.value_counts().to_dict(),
        "training_original_class_counts": target_full.answer.value_counts().to_dict(),
        "validation_sources": valid_full._src.value_counts().to_dict(), "manifest": str(manifest_path)}
    save_json(RUN_DIR / "analysis" / "data_summary.json", report)
    print(json.dumps(report, ensure_ascii=False, indent=2))


## 06. 비율 유지 Padding·학습용 Augmentation


기본 Gemma: 긴 변 2048 상한, native aspect. 1024 letterbox는 옵션.
Stage2는 원본+JPEG70+blur1.0+affine(이동3%, shear5°).
기존 photometric 기능도 옵션으로 보존한다. validation/test는 학습 증강을 적용하지 않는다.


In [ ]:
def load_image(path, policy=None):
    policy = policy or CFG["image"]
    with Image.open(path) as raw:
        img = ImageOps.exif_transpose(raw).convert("RGB")
    cap = policy.get("longest_side_cap")
    if cap and max(img.size) > cap:
        scale = cap / max(img.size)
        img = img.resize(tuple(max(1, round(n*scale)) for n in img.size), Image.Resampling.BICUBIC)
    box = policy.get("letterbox", {})
    if policy.get("resize_mode") == "letterbox" or box.get("enabled"):
        side = int(box.get("side", 1024))
        thumb = ImageOps.contain(img, (side,side), Image.Resampling.BICUBIC)
        canvas = Image.new("RGB", (side,side), tuple(box.get("fill_rgb", [255,255,255])))
        canvas.paste(thumb, ((side-thumb.width)//2,(side-thumb.height)//2))
        img = canvas
    return img

def augment_image(img, aug, rng):
    p = CFG["image"]["augmentation"]
    if aug == "orig":
        out = img.copy()
    elif aug == "compression":
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=p["jpeg_quality"])
        buf.seek(0)
        with Image.open(buf) as data:
            out = data.convert("RGB")
    elif aug == "blur":
        out = img.filter(ImageFilter.GaussianBlur(p["blur_radius"]))
    elif aug == "affine":
        tx = rng.uniform(-p["affine_translate_fraction"],p["affine_translate_fraction"])*img.width
        ty = rng.uniform(-p["affine_translate_fraction"],p["affine_translate_fraction"])*img.height
        sx = math.tan(math.radians(rng.uniform(-p["affine_shear_degrees"],p["affine_shear_degrees"])))
        sy = math.tan(math.radians(rng.uniform(-p["affine_shear_degrees"],p["affine_shear_degrees"])))
        out = img.transform(img.size, Image.Transform.AFFINE, (1,sx,tx,sy,1,ty),
                            resample=Image.Resampling.BICUBIC, fillcolor=tuple(p["fill_rgb"]))
    else:
        raise ValueError(f"지원 augmentation 없음: {aug}")
    old = p["legacy_photometric"]
    if old["enabled"]:
        if rng.random() < old["probability"]:
            out = ImageEnhance.Brightness(out).enhance(1+rng.uniform(-old["brightness"],old["brightness"]))
            out = ImageEnhance.Contrast(out).enhance(1+rng.uniform(-old["contrast"],old["contrast"]))
        if rng.random() < old["hue_saturation_probability"]:
            out = ImageEnhance.Color(out).enhance(rng.uniform(0.8,1.2))
        if rng.random() < old["noise_probability"]:
            noise_rng = np.random.default_rng(rng.randrange(2**32))
            arr = np.asarray(out).astype(np.float32)+noise_rng.normal(0,3,np.asarray(out).shape)
            out = Image.fromarray(np.clip(arr,0,255).astype(np.uint8))
        if rng.random() < old["jpeg_probability"]:
            buf = io.BytesIO()
            out.save(buf, format="JPEG", quality=rng.randint(old["jpeg_quality_min"],old["jpeg_quality_max"]))
            buf.seek(0)
            with Image.open(buf) as data:
                out = data.convert("RGB")
    return out

def permutation_safe(row):
    texts = " ".join(str(row.get(c,"")) for c in ["question",*CHOICES])
    return not re.search(r"(?:선지|보기|선택지)\s*[a-dA-D①②③④]|[a-dA-D]\s*(?:와|및|그리고)\s*[a-dA-D]|(?:첫|두|세|네)\s*번째\s*보기", texts)


## 07. OCR/DINO 선택 분기·해시 캐시


none/ocr/dino/ocr_dino. 기본은 꺼짐. OCR/DINO는 원본 이미지에서 추출해 보조 정보로만 넣는다.
CPU를 기본 장치로 사용해 큰 VLM과 GPU 메모리가 경쟁하지 않도록 한다.
DINO 객체 query는 프로젝트에 맞게 설정하고 실제 사용 목록/revision을 기록한다.


In [ ]:
_ocr_reader = None
_dino_pair = None
_dino_revision = None

def auxiliary_context(row):
    global _ocr_reader, _dino_pair, _dino_revision
    mode = CFG["extra_context"]["mode"]
    if mode == "none":
        return ""
    config = deepcopy(CFG["extra_context"])
    config["library_versions"] = {p: package_version(p) for p in ["easyocr","transformers"]}
    if mode in {"dino","ocr_dino"}:
        if _dino_revision is None:
            from huggingface_hub import HfApi
            saved=RUN_DIR/"auxiliary_metadata.json"
            _dino_revision=(json.loads(saved.read_text())["dino_revision"] if saved.exists() else
                config["dino"].get("revision") or HfApi().model_info(config["dino"]["model_id"]).sha)
            save_json(saved,{"dino_revision":_dino_revision,"model_id":config["dino"]["model_id"]})
        config["dino"]["resolved_revision"]=_dino_revision
    cache_id = digest({"image":row["_hash"],"config":config,"image_policy":CFG["image"]})
    folder = RUN_DIR.parent / "aux_cache"
    folder.mkdir(exist_ok=True)
    path = folder / f"{cache_id}.json"
    if path.exists():
        item = json.loads(path.read_text())
    else:
        image = load_image(row["_img"])
        item = {"image_hash":row["_hash"],"config":config,"ocr":[],"objects":[]}
        if mode in {"ocr","ocr_dino"}:
            import easyocr
            if _ocr_reader is None:
                _ocr_reader = easyocr.Reader(CFG["extra_context"]["ocr"]["languages"], gpu=config["device"]!="cpu")
            for box,text,confidence in _ocr_reader.readtext(np.asarray(image)):
                if float(confidence) >= config["ocr"]["cache_confidence"]:
                    item["ocr"].append({"text":text,"confidence":float(confidence),"box":np.asarray(box).tolist()})
        if mode in {"dino","ocr_dino"}:
            import torch
            from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
            if _dino_pair is None:
                name = config["dino"]["model_id"]
                dp = AutoProcessor.from_pretrained(name,revision=_dino_revision)
                dm = AutoModelForZeroShotObjectDetection.from_pretrained(name,revision=_dino_revision).to(config["device"]).eval()
                _dino_pair = dp,dm
            dp,dm = _dino_pair
            encoded = dp(images=image,text=config["dino"]["object_queries"],return_tensors="pt").to(config["device"])
            with torch.inference_mode():
                result = dm(**encoded)
            post = dp.post_process_grounded_object_detection
            threshold_name = "threshold" if "threshold" in inspect.signature(post).parameters else "box_threshold"
            kwargs = {threshold_name:config["dino"]["threshold"],"text_threshold":0.25,
                      "target_sizes":[image.size[::-1]]}
            out = post(result,encoded.input_ids,**kwargs)[0]
            labels = out.get("text_labels",out.get("labels"))
            for box,score,label in zip(out["boxes"],out["scores"],labels):
                item["objects"].append({"label":str(label),"score":float(score),"box":box.tolist()})
            item["dino_resolved_revision"] = getattr(dm.config,"_commit_hash",None)
        save_json(path,item)
    ocr = [x["text"] for x in item["ocr"] if x["confidence"]>=config["ocr"]["prompt_confidence"]][:config["ocr"]["max_items"]]
    objects = []
    for obj in sorted(item["objects"],key=lambda x:-x["score"]):
        if obj["label"] not in objects:
            objects.append(obj["label"])
    parts=[]
    if ocr:
        parts.append("OCR 참고: " + " | ".join(ocr))
    if objects:
        parts.append("객체 참고: " + ", ".join(objects[:config["dino"]["max_items"]]))
    return "\n".join(parts)

def package_version(name):
    from importlib.metadata import version, PackageNotFoundError
    try:
        return version(name)
    except PackageNotFoundError:
        return None


## 08. 프롬프트 반복·채팅 템플릿·답 위치


질문·4보기·출력 지시만 2회 반복한다. 이미지는 한 번. OCR/DINO는 참고 정보로 한 번 추가한다.
Gemma thought prefix는 필요한 경우만 붙이고 token boundary를 13에서 검사한다.


In [ ]:
def messages_for(row, image, order):
    body = str(row["question"])+"\n"+"\n".join(f"({c}) {row[CHOICES[j]]}" for c,j in zip(CHOICES,order))
    body += "\n\n"+CFG["prompt"]["instruction"]
    repeated = CFG["prompt"]["repeat_separator"].join([body]*CFG["prompt"]["repeat"])
    aux = auxiliary_context(row)
    if aux:
        repeated += "\n\n이미지가 가장 중요한 근거입니다. 아래 참고 정보가 틀리거나 이미지와 충돌하면 무시하세요.\n"+aux
    return [{"role":"system","content":CFG["prompt"]["system"]},
            {"role":"user","content":[{"type":"image","image":image},{"type":"text","text":repeated}]}]

def render_prompt(row,image,order):
    text = processor.apply_chat_template(messages_for(row,image,order), tokenize=False,
                add_generation_prompt=True,**CFG["prompt"]["template_kwargs"])
    prefix=CFG["prompt"]["answer_prefix"]
    if prefix and not text.endswith(prefix):
        text += prefix
    return text


## 09. 모델·프로세서·양자화 로드


기본 Gemma는 참고 코드의 FastVisionModel을 사용한다. 기존 HF/Agnes 프로필과
Qwen GPTQ AUTO_TRAINABLE 로더도 분리한다. GPTQ 과거 patch는 명시 옵션이고
해당 함수 signature가 맞을 때만 적용한다. 실제 backend 성공 여부는 13의 backward로 확인한다.


In [ ]:
def find_lora_targets(base):
    vision_words=("vision","visual","projector","merger","audio","embed_vision")
    suffixes=["q_proj","k_proj","v_proj","o_proj","in_proj_qkv","in_proj_z","out_proj"]
    if CFG["model"]["backend"] == "transformers_peft_delta_targets":
        suffixes += ["in_proj_b","in_proj_a"]
    if CFG["lora"]["finetune_mlp_modules"]:
        suffixes += ["gate_proj","up_proj","down_proj"]
    names=[]
    for name,module in base.named_modules():
        if any(s in name.lower() for s in vision_words) or name.endswith("lm_head"):
            continue
        if name.split(".")[-1] in suffixes and (hasattr(module,"weight") or hasattr(module,"qweight")):
            names.append(name)
    if not names:
        raise ValueError("LoRA 대상 module 없음. 모델의 named_modules를 확인하세요.")
    return sorted(set(names))

def load_vqa_model():
    global torch, F, Trainer, TrainingArguments, model, processor, tokenizer, CHOICE_IDS
    import torch
    import torch.nn.functional as F
    if not torch.cuda.is_available():
        raise RuntimeError("GPU 실행 모드는 CUDA GPU가 필요합니다. prepare로 데이터부터 확인하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("현재 기본은 BF16 지원 GPU가 필요합니다.")
    import transformers
    from transformers import AutoProcessor, AutoModelForImageTextToText, Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from huggingface_hub import HfApi
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
    spec=CFG["model"]
    existing_meta=RUN_DIR/"model_metadata.json"
    previous_revision=json.loads(existing_meta.read_text()).get("resolved_revision") if existing_meta.exists() else None
    if previous_revision and spec["revision"] and previous_revision!=spec["revision"]:
        raise ValueError("재개 run의 model revision이 다릅니다.")
    revision=spec["revision"] or previous_revision or HfApi().model_info(spec["model_id"]).sha
    kwargs={"revision":revision,"trust_remote_code":spec["trust_remote_code"]}
    if spec["backend"] == "unsloth_fastvision":
        from unsloth import FastVisionModel
        model,processor=FastVisionModel.from_pretrained(spec["model_id"],
            load_in_4bit=spec["load_in_4bit"], max_seq_length=spec["max_seq_length"],
            use_gradient_checkpointing=spec["gradient_checkpointing"],**kwargs)
        model=FastVisionModel.get_peft_model(model,
            finetune_vision_layers=CFG["lora"]["finetune_vision_layers"],
            finetune_language_layers=CFG["lora"]["finetune_language_layers"],
            finetune_attention_modules=CFG["lora"]["finetune_attention_modules"],
            finetune_mlp_modules=CFG["lora"]["finetune_mlp_modules"],
            r=CFG["lora"]["r"], lora_alpha=CFG["lora"]["alpha"],
            lora_dropout=CFG["lora"]["dropout"], bias=CFG["lora"]["bias"],
            random_state=SEED,use_gradient_checkpointing=spec["gradient_checkpointing"])
    elif spec["backend"] == "gptq_peft_with_backward_check":
        from gptqmodel import GPTQModel,BACKEND
        processor=AutoProcessor.from_pretrained(spec["model_id"],**kwargs)
        wrapper=GPTQModel.load(spec["model_id"],backend=BACKEND.AUTO_TRAINABLE,
                              dtype=torch.bfloat16,**kwargs)
        base=wrapper.model if hasattr(wrapper,"model") else wrapper
        base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,
                            gradient_checkpointing_kwargs={"use_reentrant":False})
        if ENABLE_GPTQ_TRITON_LEGACY_PATCH:
            from torch.amp import custom_bwd
            from gptqmodel.nn_modules.triton_utils import dequant
            function=dequant.QuantLinearFunction
            if len(inspect.signature(function.forward).parameters) != 9:
                raise RuntimeError("과거 8-input GPTQ patch와 현재 forward signature가 다릅니다.")
            @custom_bwd(device_type="cuda")
            def fixed_backward(ctx,grad_output):
                qweight,scales,qzeros,g_idx=ctx.saved_tensors
                grad_input=None
                if ctx.needs_input_grad[0]:
                    grad_input=dequant.quant_matmul(grad_output,qweight,scales,qzeros,g_idx,
                        ctx.bits,ctx.pack_bits,ctx.maxq,transpose=True)
                return grad_input,None,None,None,None,None,None,None
            function.backward=staticmethod(fixed_backward)
        names=find_lora_targets(base)
        model=get_peft_model(base,LoraConfig(r=CFG["lora"]["r"],lora_alpha=CFG["lora"]["alpha"],
            lora_dropout=CFG["lora"]["dropout"],bias="none",target_modules=names,task_type="CAUSAL_LM"))
    else:
        processor=AutoProcessor.from_pretrained(spec["model_id"],**kwargs)
        quant={}
        if spec["load_in_4bit"]:
            from transformers import BitsAndBytesConfig
            quant["quantization_config"]=BitsAndBytesConfig(load_in_4bit=True,
                bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.bfloat16)
        base=AutoModelForImageTextToText.from_pretrained(spec["model_id"],dtype=torch.bfloat16,
                device_map={"":0},attn_implementation="sdpa",**kwargs,**quant)
        if spec["load_in_4bit"]:
            base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,
                    gradient_checkpointing_kwargs={"use_reentrant":False})
        else:
            base.requires_grad_(False)
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
            base.enable_input_require_grads()
        names=find_lora_targets(base)
        model=get_peft_model(base,LoraConfig(r=CFG["lora"]["r"],lora_alpha=CFG["lora"]["alpha"],
            lora_dropout=CFG["lora"]["dropout"],bias="none",target_modules=names,task_type="CAUSAL_LM"))
    model.config.use_cache=False
    tokenizer=getattr(processor,"tokenizer",processor)
    tokenizer.padding_side="left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token=tokenizer.eos_token
    ip=getattr(processor,"image_processor",None)
    budget=CFG["image"]["max_soft_tokens"]
    if PRESET=="gemma31_legacy":
        budget=560
    if budget:
        for obj in [ip,processor,model.config,getattr(model.config,"vision_config",None)]:
            for name in ["max_soft_tokens","image_seq_length","vision_soft_tokens_per_image","default_output_length"]:
                if obj is not None and hasattr(obj,name):
                    try:
                        setattr(obj,name,budget)
                    except (AttributeError,TypeError):
                        pass
    if PRESET in {"qwen27_only","team_common","agnes_legacy"} and ip is not None:
        high=1024*1024 if PRESET!="agnes_legacy" else 560*32*32
        low=256*28*28 if PRESET=="team_common" else 64*32*32
        for obj in [ip,processor]:
            for name,value in [("min_pixels",low),("max_pixels",high)]:
                if hasattr(obj,name):
                    try:
                        setattr(obj,name,value)
                    except (AttributeError,TypeError):
                        pass
        size=getattr(ip,"size",None)
        if isinstance(size,dict):
            ip.size={**size,"shortest_edge":low,"longest_edge":high}
        elif size is not None and hasattr(size,"shortest_edge") and hasattr(size,"longest_edge"):
            size.shortest_edge=low
            size.longest_edge=high
    CHOICE_IDS=[]
    for c in CHOICES:
        ids=tokenizer.encode(c,add_special_tokens=False)
        if len(ids)!=1 or tokenizer.decode(ids)!=c:
            raise RuntimeError(f"정답 글자 단일 토큰 검증 실패: {c}: {ids}")
        CHOICE_IDS.append(ids[0])
    save_json(RUN_DIR/"model_metadata.json",{"model_id":spec["model_id"],"resolved_revision":revision,
        "observed_config_revision":getattr(model.config,"_commit_hash",None),
        "choice_token_ids":dict(zip(CHOICES,CHOICE_IDS)),"image_processor":str(ip),
        "backend":spec["backend"],"budget":budget})
    model.print_trainable_parameters()
    props=torch.cuda.get_device_properties(0)
    print("GPU:",props.name,"| VRAM GiB:",round(props.total_memory/2**30,2))
    environment={"python":sys.version,"platform":platform.platform(),"cuda":torch.version.cuda,
        "gpu":props.name,"vram_bytes":props.total_memory,
        "versions":{p:package_version(p) for p in ["torch","torchvision","unsloth","transformers","peft",
                      "bitsandbytes","accelerate","datasets","gptqmodel"]}}
    save_json(RUN_DIR/"environment.json",environment)
    (RUN_DIR/"environment_freeze.txt").write_text(subprocess.check_output(
        [sys.executable,"-m","pip","freeze"],text=True),encoding="utf-8")
    if ADAPTER_PATH:
        from peft import set_peft_model_state_dict
        from peft.utils import load_peft_weights
        config=Path(ADAPTER_PATH)/"adapter_config.json"
        if not config.is_file():
            raise FileNotFoundError("adapter_config.json 없음")
        adapter_spec=json.loads(config.read_text())
        if adapter_spec.get("r")!=CFG["lora"]["r"] or adapter_spec.get("lora_alpha")!=CFG["lora"]["alpha"]:
            raise ValueError("불러올 adapter의 r/alpha와 설정이 다릅니다.")
        set_peft_model_state_dict(model,load_peft_weights(str(ADAPTER_PATH)))
    elif RUN_MODE=="infer":
        if not RESUME_RUN_DIR or not (RUN_DIR/"adapter"/"adapter_config.json").is_file():
            raise ValueError("infer 모드에는 완성 ADAPTER_PATH가 필요합니다.")
        from peft import set_peft_model_state_dict
        from peft.utils import load_peft_weights
        set_peft_model_state_dict(model,load_peft_weights(str(RUN_DIR/"adapter")))

def set_model_mode(training):
    if CFG["model"]["backend"]=="unsloth_fastvision":
        from unsloth import FastVisionModel
        (FastVisionModel.for_training if training else FastVisionModel.for_inference)(model)
    else:
        model.train(training)
    model.config.use_cache=False
    if CFG["model"]["backend"]=="gptq_peft_with_backward_check":
        for module in model.modules():
            if module.__class__.__name__=="TritonV2Linear":
                module.eval()

if GPU_RUN:
    load_vqa_model()


## 10. LoRA 대상·학습 범위 감사


r/alpha뿐 아니라 실제 주입 module 이름, trainable parameter, vision/lm_head 제외를 기록한다.
기존 프로필은 기존 attention 계열 target를 사용하되 새 공통 loss·분할·저장 경로를 적용한다.
과거 성적을 낸 실행의 완전한 복원이라고 표현하지 않는다.


In [ ]:
def audit_lora():
    names=[n for n,m in model.named_modules() if hasattr(m,"lora_A")]
    trainable=[(n,p.numel()) for n,p in model.named_parameters() if p.requires_grad]
    if not names or not trainable:
        raise RuntimeError("주입된 LoRA/학습 파라미터가 없습니다.")
    if any("lora_" not in n for n,_ in trainable):
        raise RuntimeError("LoRA 외 파라미터가 학습 가능 상태입니다. 동결 정책 확인 필요.")
    if not CFG["lora"]["finetune_vision_layers"] and any(
            any(w in n.lower() for w in ["vision","visual","projector"]) for n in names):
        raise RuntimeError("vision 고정 설정과 LoRA target가 충돌합니다.")
    save_json(RUN_DIR/"analysis"/"lora_modules.json",{"modules":names,"trainable":trainable,
              "trainable_count":sum(v for _,v in trainable),"lora_config":CFG["lora"]})

if GPU_RUN:
    audit_lora()


## 11. 학습 샘플·정답 위치 균형·1토큰 loss 입력


먼저 원본×증강종류의 학습 계획을 만든다. 안전하게 재배치할 수 있는 샘플은
a/b/c/d 목표 위치를 균등 배정한다. 보기와 정답을 같이 재매핑한다.
위치 참조 문항/고정 반전 문항은 예외로 기록하며 실제 분포를 저장한다.


In [ ]:
def training_plan(frame,augs):
    rows=frame.to_dict("records")
    if CFG["data"]["hard_reverse"]["enabled"]:
        if "is_hard" not in frame:
            raise ValueError("hard reverse에는 검토로 지정한 is_hard 열이 필요합니다.")
        additional=[]
        for r in rows:
            if str(r.get("is_hard","")).strip().lower() in {"true","1","yes"}:
                if not permutation_safe(r):
                    continue
                new=dict(r)
                new["id"]=r["id"]+"__rev"
                for j,c in enumerate(CHOICES):
                    new[c]=r[CHOICES[3-j]]
                new["answer"]=CHOICES[3-CHOICES.index(r["answer"])]
                new["_fixed_order"]=True
                additional.append(new)
        rows+=additional
    entries=[{"row":r,"aug":aug,"safe":permutation_safe(r) and not r.get("_fixed_order",False)}
             for r in rows for aug in augs]
    n=len(entries)
    rng=random.Random(SEED)
    eligible=[i for i,e in enumerate(entries) if e["safe"]]
    rng.shuffle(eligible)
    assigned={i:CHOICES.index(e["row"]["answer"]) for i,e in enumerate(entries) if not e["safe"]}
    count=np.bincount(list(assigned.values()),minlength=4)
    mode=CFG["data"]["training_option_order"]["mode"]
    if mode=="balanced_target_positions":
        quota=np.array([n//4+int(j<n%4) for j in range(4)])
        for i in eligible:
            deficit=quota-count
            candidates=np.flatnonzero(deficit==deficit.max()).tolist()
            target=rng.choice(candidates)
            assigned[i]=target
            count[target]+=1
    for i,e in enumerate(entries):
        gold=CHOICES.index(e["row"]["answer"])
        order=list(range(4))
        sample_rng=random.Random(SEED*1_000_003+i)
        if e["safe"]:
            if mode=="balanced_target_positions":
                others=[j for j in range(4) if j!=gold]
                sample_rng.shuffle(others)
                order=others
                order.insert(assigned[i],gold)
            elif mode=="random_shuffle":
                sample_rng.shuffle(order)
            elif mode!="original":
                raise ValueError("선지 순서 mode 오류")
        e["order"]=order
        e["gold_index"]=order.index(gold)
        e["seed"]=SEED*1_000_003+i
    return entries

class ChoiceDataset:
    def __init__(self,frame,augs):
        self.plan=training_plan(frame,augs)
    def __len__(self):
        return len(self.plan)
    def __getitem__(self,index):
        e=self.plan[index]
        image=augment_image(load_image(e["row"]["_img"]),e["aug"],random.Random(e["seed"]))
        return {"text":render_prompt(e["row"],image,e["order"]),"image":image,
                "target":CHOICE_IDS[e["gold_index"]]}

def audit_training_plan(dataset,name):
    records=[{"id":e["row"]["id"],"source":e["row"]["_src"],"aug":e["aug"],
              "permutation_safe":e["safe"],"order":json.dumps(e["order"]),
              "answer_after":CHOICES[e["gold_index"]]} for e in dataset.plan]
    pd.DataFrame(records).to_csv(RUN_DIR/"analysis"/f"training_plan_{name}.csv",index=False)
    counts=Counter(r["answer_after"] for r in records)
    save_json(RUN_DIR/"analysis"/f"position_balance_{name}.json",{"counts":counts,
        "ratios":{c:counts[c]/len(records) for c in CHOICES},
        "fixed_or_unsafe":sum(not e["safe"] for e in dataset.plan),"n":len(records)})
    print(name,"학습 답 위치:",dict(counts))

if AUTO_RUN and RUN_MODE=="prepare":
    cpu_dataset=ChoiceDataset(target_df,CFG["training"]["stages"][-1]["augmentation"])
    audit_training_plan(cpu_dataset,"target")


## 12. Processor·Collator·DataLoader


프로세서는 실제 이미지와 text를 함께 받는다. left padding으로 마지막 위치를 답 위치로 고정한다.
학습 target은 단일 token ID이다. context 초과 시 질문/이미지를 조용히 자르지 않고 중단한다.


In [ ]:
_images_nested=None
_logits_to_keep=None

def call_processor(texts,images):
    global _images_nested
    kwargs=dict(text=texts,padding=True,return_tensors="pt",add_special_tokens=False)
    if _images_nested is None:
        failures=[]
        for nested in [False,True]:
            try:
                batch=processor(images=[[im] for im in images] if nested else images,**kwargs)
                _images_nested=nested
                break
            except (TypeError,ValueError) as exc:
                failures.append(str(exc))
        else:
            raise ValueError("processor image 구조 오류: "+" | ".join(failures))
    else:
        batch=processor(images=[[im] for im in images] if _images_nested else images,**kwargs)
    lengths=batch["attention_mask"].sum(-1)
    if lengths.max().item()>CFG["model"]["max_seq_length"]:
        raise ValueError(f"context 초과 {lengths.max().item()}; prompt/image budget을 조정하세요.")
    return dict(batch)

def collate_choices(features):
    encoded=call_processor([f["text"] for f in features],[f["image"] for f in features])
    encoded["target_ids"]=torch.tensor([f["target"] for f in features],dtype=torch.long)
    return encoded

def to_device(batch):
    return {k:v.to(model.device) if torch.is_tensor(v) else v for k,v in batch.items()}

def last_logits(batch,current_model=None):
    global _logits_to_keep
    current_model=current_model if current_model is not None else model
    if _logits_to_keep is not False:
        try:
            output=current_model(**batch,logits_to_keep=1,use_cache=False)
            _logits_to_keep=True
            return output.logits[:,-1,:]
        except TypeError as exc:
            if "logits_to_keep" not in str(exc):
                raise
            _logits_to_keep=False
    return current_model(**batch,use_cache=False).logits[:,-1,:]


## 13. 학습 전 점검·실제 backward·VRAM


학습 전에 답 token boundary, processor 입력, 실제 forward/backward, 유한 loss/gradient를 점검한다.
점검 backward는 optimizer step을 하지 않아 가중치를 바꾸지 않는다.
prepare는 CPU 데이터 점검까지 수행하고 모델을 불러오지 않는다.


In [ ]:
def gpu_preflight():
    row=valid_df.iloc[0].to_dict()
    image=load_image(row["_img"])
    text=render_prompt(row,image,list(range(4)))
    before=tokenizer.encode(text,add_special_tokens=False)
    for c,token_id in zip(CHOICES,CHOICE_IDS):
        after=tokenizer.encode(text+c,add_special_tokens=False)
        if after[:-1]!=before or after[-1]!=token_id:
            raise RuntimeError(f"다음 답 위치와 글자 token 경계 불일치: {c}")
    dataset=ChoiceDataset(target_df.head(2),["orig"])
    batch=to_device(collate_choices([dataset[0]]))
    targets=batch.pop("target_ids")
    torch.cuda.reset_peak_memory_stats()
    set_model_mode(True)
    model.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(RUN_MODE!="infer"),torch.autocast("cuda",dtype=torch.bfloat16):
        logits=last_logits(batch)
        loss=F.cross_entropy(logits.float(),targets)
    if not torch.isfinite(loss):
        raise RuntimeError("preflight loss가 유한하지 않습니다.")
    gradients=[]
    if RUN_MODE!="infer":
        loss.backward()
        gradients=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not gradients or any(not torch.isfinite(g).all() for g in gradients):
            raise RuntimeError("실제 backward 실패: LoRA gradient 없음/NaN")
    model.zero_grad(set_to_none=True)
    set_model_mode(False)
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
        p=torch.softmax(last_logits(batch).float(),dim=-1)
        mass=float(p[:,CHOICE_IDS].sum(-1).mean())
    result={"loss":float(loss.detach()),"choice_probability_mass":mass,
        "prompt_probe_text_tokens":len(before),"training_batch_input_tokens":int(batch["attention_mask"].sum().item()),
        "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
        "backward_passed":True if RUN_MODE!="infer" else None,"optimizer_step_performed":False}
    save_json(RUN_DIR/"analysis"/"preflight.json",result)
    print(result)
    del batch,logits,loss,gradients,p
    gc.collect()
    torch.cuda.empty_cache()

if GPU_RUN:
    gpu_preflight()
elif AUTO_RUN:
    print("prepare: CPU 데이터 검증 완료. GPU 학습/추론은 생략합니다.")


## 14. 외부 데이터 구성·2-Stage 파인튜닝·Checkpoint


Stage1: MTVQA 한국어 4지선다 변환+KRETA, 원본 이미지 1 epoch.
Stage2: 내부 train+선별 dev, 4종 이미지 1 epoch. 같은 adapter를 이어 학습하며 optimizer는 새로 만든다.
아래는 단계/Trainer 정의다. 실제 실행은 19에서 한다. smoke는 기본 내부 소량만 검증한다.
외부 test split은 외부 학습 데이터이며 대회 test 답을 사용하는 것이 아니다.


In [ ]:
EXTERNAL_ENABLED = CFG["external_data"]["enabled"] and RUN_MODE!="infer" and (RUN_MODE!="smoke" or SMOKE_USE_EXTERNAL)
EXT_DIR = Path(CFG["run"]["output_root"]) / "external_cache"
KRETA_REPO=CFG["external_data"]["kreta"]["repo_id"]
KRETA_SPLIT=CFG["external_data"]["kreta"]["split"]
MTVQA_REPO=CFG["external_data"]["mtvqa"]["repo_id"]
MTVQA_SPLITS=CFG["external_data"]["mtvqa"]["splits"]
MTVQA_LANGS=CFG["external_data"]["mtvqa"]["languages"]
MTVQA_MAX_ANSWER_CHARS=CFG["external_data"]["mtvqa"]["max_answer_chars"]
MTVQA_DISTRACTOR_POOL=CFG["external_data"]["mtvqa"]["distractor_pool"]
EXTERNAL_REVISIONS={}

def safe_name(s):
    return re.sub(r"[^0-9A-Za-z_\-]","_",str(s))


In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

def load_kreta():
    cache = EXT_DIR / "kreta" / "kreta.csv"
    img_dir = EXT_DIR / "kreta" / "images"
    if not cache.exists():
        import base64
        from collections import Counter
        from datasets import load_dataset
        img_dir.mkdir(parents=True, exist_ok=True)
        rows, skipped, seen = [], Counter(), set()
        for r in tqdm(load_dataset(KRETA_REPO, split=KRETA_SPLIT, revision=EXTERNAL_REVISIONS["kreta"]), desc="KRETA"):
            opts = [r.get(k) for k in "ABCD"]
            ans = str(r.get("answer") or "").strip().upper()
            if any(o is None or str(o).strip() in ("", "nan", "None") for o in opts):
                skipped["missing_option"] += 1; continue
            if ans not in ("A", "B", "C", "D"):
                skipped["bad_answer"] += 1; continue
            if len({str(o).strip() for o in opts}) < 4:
                skipped["duplicate_option"] += 1; continue
            rid = safe_name(r["id"])
            while rid in seen:
                rid += "_"
            seen.add(rid)
            p = img_dir / f"{rid}.jpg"
            if not p.exists():
                b64 = str(r["image"])
                if b64.startswith("data:"):
                    b64 = b64.split(",", 1)[1]
                Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB").save(p, quality=95)
            rows.append({"id": f"kreta_{rid}", "path": str(p.relative_to(EXT_DIR)),
                         "question": str(r["question"]).strip(),
                         **{c: str(o).strip() for c, o in zip(CHOICES, opts)}, "answer": ans.lower(),
                         "topic_difficulty": r.get("topic_difficulty"), "image_type": r.get("image_type")})
        pd.DataFrame(rows).to_csv(cache, index=False, encoding="utf-8-sig")
        print(f"[kreta] 저장 {len(rows)} | 제외 {dict(skipped)}")
    return pd.read_csv(cache, dtype=str, keep_default_na=False, encoding="utf-8-sig")


def mt_norm(s):
    return re.sub(r"[\s\W_]+", "", str(s)).lower()


def has_digit(s):
    return any(ch.isdigit() for ch in str(s))


def load_mtvqa_qa():
    cache = EXT_DIR / "mtvqa" / "mtvqa_ko_qa.csv"
    img_dir = EXT_DIR / "mtvqa" / "images"
    if not cache.exists():
        import ast
        from datasets import load_dataset
        img_dir.mkdir(parents=True, exist_ok=True)
        langs = {l.upper() for l in MTVQA_LANGS}
        recs = []
        for split in MTVQA_SPLITS:
            ds = load_dataset(MTVQA_REPO, split=split, revision=EXTERNAL_REVISIONS["mtvqa"])
            idx = [i for i, l in enumerate(ds["lang"]) if str(l).upper() in langs]
            print(f"[mtvqa] {split}: {len(idx)} images")
            for r in tqdm(ds.select(idx), desc=f"MTVQA {split}"):
                img_id = f"{split}_{safe_name(r['id'])}"
                p = img_dir / f"{img_id}.jpg"
                if not p.exists():
                    r["image"].convert("RGB").save(p, quality=95)
                qa = r["qa_pairs"]
                if isinstance(qa, str):
                    qa = ast.literal_eval(qa)
                for k, item in enumerate(qa):
                    recs.append({"img_id": img_id, "k": k, "path": str(p.relative_to(EXT_DIR)),
                                 "question": " ".join(str(item.get("question", "")).split()),
                                 "answer_text": " ".join(str(item.get("answer", "")).split())})
        pd.DataFrame(recs).to_csv(cache, index=False, encoding="utf-8-sig")
    return pd.read_csv(cache, dtype=str, keep_default_na=False, encoding="utf-8-sig")


def build_mtvqa_mc(qa):
    """주관식 → 4지선다. 오답: 같은 이미지의 다른 답 → 같은 유형(숫자 포함 여부)·비슷한 길이의 답."""
    n0 = len(qa)
    qa = qa[(qa["question"].str.len() > 0) & (qa["answer_text"].str.len() > 0)
            & (qa["answer_text"].str.len() <= MTVQA_MAX_ANSWER_CHARS)].copy()
    qa["norm"] = qa["answer_text"].map(mt_norm)
    qa = qa[qa["norm"].str.len() > 0].reset_index(drop=True)
    uniq = qa.drop_duplicates("norm")
    pool = {True: [], False: []}
    for a, n in zip(uniq["answer_text"], uniq["norm"]):
        pool[has_digit(a)].append((a, n, len(a)))
    by_img = qa.groupby("img_id")["answer_text"].apply(list).to_dict()
    ok = lambda n, used, g: n and n not in used and n not in g and g not in n

    rows = []
    for i, r in enumerate(qa.to_dict("records")):
        rng = random.Random(SEED * 7919 + i)
        gold, gn = r["answer_text"], r["norm"]
        opts, used = [gold], {gn}
        same = list(by_img[r["img_id"]]); rng.shuffle(same)
        for a in same:
            n = mt_norm(a)
            if ok(n, used, gn):
                opts.append(a); used.add(n)
            if len(opts) == 4:
                break
        if len(opts) < 4:
            cands = sorted((c for c in pool[has_digit(gold)] if ok(c[1], used, gn)), key=lambda c: abs(c[2] - len(gold)))
            near = cands[:MTVQA_DISTRACTOR_POOL]; rng.shuffle(near)
            for a, n, _ in near:
                if n not in used:
                    opts.append(a); used.add(n)
                if len(opts) == 4:
                    break
        if len(opts) < 4:
            continue
        order = [0, 1, 2, 3]; rng.shuffle(order)
        rows.append({"id": f"mtvqa_{r['img_id']}_{r['k']}", "path": r["path"], "question": r["question"],
                     **dict(zip(CHOICES, [opts[j] for j in order])), "answer": CHOICES[order.index(0)]})
    print(f"[mtvqa] QA {n0} → 4지선다 {len(rows)}")
    return pd.DataFrame(rows)


In [ ]:
def prepare_stages():
    global EXT_DIR,EXTERNAL_REVISIONS
    stages=[]
    if EXTERNAL_ENABLED:
        from huggingface_hub import HfApi
        api=HfApi()
        revision=CFG["external_data"]["dataset_revision"]
        previous=RUN_DIR/"external_metadata.json"
        if previous.exists():
            EXTERNAL_REVISIONS=json.loads(previous.read_text())["revisions"]
        else:
            EXTERNAL_REVISIONS={"kreta":revision or api.dataset_info(KRETA_REPO).sha,
                                "mtvqa":revision or api.dataset_info(MTVQA_REPO).sha}
        EXT_DIR=EXT_DIR/digest({"settings":CFG["external_data"],"revisions":EXTERNAL_REVISIONS,"seed":SEED})[:16]
        EXT_DIR.mkdir(parents=True,exist_ok=True)
        external_parts=[]
        for name,loader in [("mtvqa",lambda:build_mtvqa_mc(load_mtvqa_qa())),("kreta",load_kreta)]:
            frame=loader().copy()
            if frame.empty:
                raise RuntimeError(f"{name} 외부 변환 결과가 비어 있습니다.")
            frame["_img"]=[resolve_image(p,EXT_DIR/"rows.csv",EXT_DIR) for p in frame.path]
            frame["_hash"]=[hash_file(p) for p in frame._img]
            frame["_src"]=name
            frame=remove_leakage(frame,valid_full,name)
            external_parts.append(frame)
        ext=pd.concat(external_parts,ignore_index=True)
        if ext.id.duplicated().any() or not ext.answer.isin(CHOICES).all():
            raise ValueError("외부 데이터 ID/정답 오류")
        ext.to_csv(RUN_DIR/"external_rows.csv",index=False,encoding="utf-8-sig")
        save_json(RUN_DIR/"external_metadata.json",{"revisions":EXTERNAL_REVISIONS,
            "n_rows":len(ext),"split_use":"external benchmark train/test used as training",
            "cache_dir":str(EXT_DIR)})
        if RUN_MODE=="smoke":
            ext=ext.head(SMOKE_BASE_ROWS)
        stage=deepcopy(CFG["training"]["stages"][0])
        stage["df"]=ext
        stages.append(stage)
    stage=deepcopy(CFG["training"]["stages"][-1])
    stage["df"]=target_df
    stages.append(stage)
    for s in stages:
        s["dataset"]=ChoiceDataset(s["df"],s["augmentation"])
        if not len(s["dataset"]):
            raise RuntimeError(f"{s['name']} 학습 데이터 없음")
        audit_training_plan(s["dataset"],s["name"])
    return stages

def training_arguments(stage,output,max_steps=None,save=True):
    tc=CFG["training"]
    steps=math.ceil(math.ceil(len(stage["dataset"])/tc["per_device_train_batch_size"])/tc["gradient_accumulation_steps"]*stage["epochs"])
    if max_steps is None:
        max_steps=SMOKE_MAX_STEPS if RUN_MODE=="smoke" else -1
    warmup=max(1,round(tc["warmup_ratio"]*(max_steps if max_steps>0 else steps)))
    return TrainingArguments(output_dir=str(output),
        per_device_train_batch_size=tc["per_device_train_batch_size"],
        gradient_accumulation_steps=tc["gradient_accumulation_steps"],
        num_train_epochs=stage["epochs"],max_steps=max_steps,learning_rate=stage["learning_rate"],
        lr_scheduler_type=tc["scheduler"],warmup_steps=warmup,weight_decay=tc["weight_decay"],
        optim=tc["optimizer"],max_grad_norm=tc["max_grad_norm"],bf16=True,tf32=True,
        logging_steps=1 if RUN_MODE=="smoke" else tc["logging_steps"],
        save_strategy="steps" if save else "no",save_steps=tc["save_steps"],
        save_total_limit=tc["save_total_limit"],report_to="none",seed=SEED,data_seed=SEED,
        remove_unused_columns=False,dataloader_num_workers=tc["num_workers"],
        gradient_checkpointing=False)

def make_trainer(stage,args):
    class AnswerTokenTrainer(Trainer):
        def __init__(self,*a,**k):
            super().__init__(*a,**k)
            self.model_accepts_loss_kwargs=False
        def compute_loss(self,current_model,inputs,return_outputs=False,num_items_in_batch=None):
            data=dict(inputs)
            target=data.pop("target_ids")
            logits=last_logits(data,current_model)
            loss=F.cross_entropy(logits.float(),target)
            return (loss,{"logits":logits[:,None,:]}) if return_outputs else loss
    return AnswerTokenTrainer(model=model,args=args,train_dataset=stage["dataset"],data_collator=collate_choices)

def adapter_fingerprint(path):
    folder=Path(path)
    files=sorted(folder.glob("adapter_model*"))+[folder/"adapter_config.json"]
    if not files or not all(f.is_file() for f in files):
        raise FileNotFoundError(f"adapter 파일 없음: {folder}")
    return digest({f.name:hash_file(f) for f in files})

def train_one_stage(stage):
    from transformers.trainer_utils import get_last_checkpoint
    from peft import set_peft_model_state_dict
    from peft.utils import load_peft_weights
    folder=RUN_DIR/stage["save_adapter"]
    marker=folder/"done.json"
    checkpoints=RUN_DIR/"checkpoints"/stage["name"]
    stage_signature=digest({"config":CONFIG_SIGNATURE,"data":data_fingerprint,"name":stage["name"],
        "model_revision":json.loads((RUN_DIR/"model_metadata.json").read_text())["resolved_revision"],
        "rows":stage["df"][["id","_hash","question",*CHOICES,"answer"]].to_dict("records"),
        "n_samples":len(stage["dataset"]),"epochs":stage["epochs"],"lr":stage["learning_rate"]})
    if marker.exists():
        result=json.loads(marker.read_text())
        if result["signature"]!=stage_signature or result["adapter_hash"]!=adapter_fingerprint(folder):
            raise ValueError(f"{stage['name']} 완료 marker/adapter가 달라졌습니다.")
        set_peft_model_state_dict(model,load_peft_weights(str(folder)))
        print("완료 단계 재사용:",stage["name"])
        return result
    set_model_mode(True)
    trainer=make_trainer(stage,training_arguments(stage,checkpoints))
    last=get_last_checkpoint(str(checkpoints)) if checkpoints.exists() else None
    before=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    trained=trainer.train(resume_from_checkpoint=last)
    folder.mkdir(parents=True,exist_ok=True)
    model.save_pretrained(folder)
    processor.save_pretrained(folder)
    pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR/"logs"/f"{stage['name']}_history.csv",index=False)
    selection={"policy":CFG["training"]["checkpoint_selection"],"selected":"final_adapter"}
    if stage["name"]=="target" and selection["policy"]=="best_retained_checkpoint":
        records=[]
        candidates=sorted(checkpoints.glob("checkpoint-*"))+[folder]
        for candidate in candidates:
            if not (candidate/"adapter_config.json").is_file():
                continue
            set_peft_model_state_dict(model,load_peft_weights(str(candidate)))
            _,probs=evaluate_in_memory(valid_df,CFG["inference"]["shifts"])
            accuracy=float((probs.mean(0).argmax(1)==valid_df.answer.map(CHOICES.index).to_numpy()).mean())
            records.append({"path":str(candidate),"accuracy":accuracy})
        best=max(records,key=lambda r:r["accuracy"])
        set_peft_model_state_dict(model,load_peft_weights(best["path"]))
        model.save_pretrained(folder)
        processor.save_pretrained(folder)
        selection.update(selected=best["path"],records=records,selection_validation_ids=valid_df.id.tolist())
    elif selection["policy"]!="last_stage_final":
        if selection["policy"]!="best_retained_checkpoint":
            raise ValueError("checkpoint selection 정책 오류")
    result={"stage":stage["name"],"signature":stage_signature,"adapter_hash":adapter_fingerprint(folder),
        "train_loss":float(trained.training_loss),"hours":(time.perf_counter()-before)/3600,
        "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,
        "steps":trainer.state.global_step,"n_samples":len(stage["dataset"]),
        "epochs":stage["epochs"],"learning_rate":stage["learning_rate"],"selection":selection,
        "is_smoke":RUN_MODE=="smoke"}
    save_json(marker,result)
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return result

def benchmark_training(stage,steps=20):
    from peft import get_peft_model_state_dict,set_peft_model_state_dict
    saved={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}
    rng=torch.get_rng_state()
    cuda_rng=torch.cuda.get_rng_state_all()
    trainer=None
    try:
        set_model_mode(True)
        trainer=make_trainer(stage,training_arguments(stage,RUN_DIR/"analysis"/"benchmark",max_steps=steps,save=False))
        start=time.perf_counter()
        torch.cuda.reset_peak_memory_stats()
        trainer.train()
        result={"steps":steps,"seconds_per_step":(time.perf_counter()-start)/steps,
                "peak_allocated_gib":torch.cuda.max_memory_allocated()/2**30,"adapter_restored_after":True}
        save_json(RUN_DIR/"analysis"/"benchmark.json",result)
        print(result)
    finally:
        set_peft_model_state_dict(model,saved)
        torch.set_rng_state(rng)
        torch.cuda.set_rng_state_all(cuda_rng)
        model.zero_grad(set_to_none=True)
        del trainer,saved
        gc.collect()
        torch.cuda.empty_cache()


## 15. 생성 기반 답 파싱·진단


정식 확률 추론과 별도로 generate 결과를 소량 확인한다. 새로 생성한 부분만 파싱한다.
불명확한 출력은 invalid로 기록한다. 토큰 확률 대신 hard 답을 one-hot으로 만들지 않는다.


In [ ]:
def parse_answer(text):
    text=str(text).strip()
    if re.fullmatch(r"[abcd]",text,re.I):
        return text.lower()
    match=re.fullmatch(r"(?:정답\s*[:：]?\s*)?[\(\[]?([abcd])[\)\].]?",text,re.I)
    return match.group(1).lower() if match else None

def generation_diagnostic(frame,n=8):
    records=[]
    set_model_mode(False)
    for row in frame.head(n).to_dict("records"):
        image=load_image(row["_img"])
        text=render_prompt(row,image,list(range(4)))
        batch=to_device(call_processor([text],[image]))
        with torch.inference_mode():
            out=model.generate(**batch,max_new_tokens=16,do_sample=False,use_cache=False)
        generated=tokenizer.decode(out[0,batch["input_ids"].shape[1]:],skip_special_tokens=True)
        records.append({"id":row["id"],"generated":generated,"parsed":parse_answer(generated),
                        "answer":row.get("answer"),"valid":parse_answer(generated) is not None})
    return pd.DataFrame(records)


## 16. Direct Choice Logit·Softmax·확률 정의


다음 답 위치의 실제 a/b/c/d logits를 사용한다. 학습의 전체 어휘 CE와 다른 4선지 조건부 softmax이다.
각 shift 확률을 원래 보기 순서로 되돌린 뒤 평균한다. TTA 평균의 log를 실제 logits로 저장하지 않는다.


In [ ]:
def softmax_numpy(logits,temperature=1.0):
    if temperature<=0:
        raise ValueError("temperature는 양수여야 합니다.")
    z=np.asarray(logits,dtype=np.float32)/temperature
    z=z-z.max(axis=-1,keepdims=True)
    e=np.exp(z)
    return e/e.sum(axis=-1,keepdims=True)

def restore_choice_order(shown,order):
    restored=np.empty(4,dtype=np.float32)
    restored[np.asarray(order,dtype=np.int64)]=shown
    return restored

def check_probabilities(probs,n):
    p=np.asarray(probs)
    if p.shape!=(n,4) or not np.isfinite(p).all() or np.any(p<0) or not np.allclose(p.sum(-1),1,atol=1e-5):
        raise ValueError(f"확률 shape/유한값/합 검증 실패: {p.shape}")


## 17. Option Permutation 검증·오류 재시도


shift s 입력 order=[s,s+1,s+2,s+3] mod4. 안전하지 않은 위치 참조 문항은 실제 order0을 유지한다.
실제 effective order를 출력 CSV에 남긴다. OOM은 batch를 줄여 재시도하고 batch1에서도 실패하면 중단한다.


In [ ]:
def forward_shift(rows,shift,image_policy=None):
    orders=[[(j+shift)%4 for j in range(4)] if permutation_safe(r) else list(range(4)) for r in rows]
    images=[load_image(r["_img"],image_policy) for r in rows]
    texts=[render_prompt(r,img,order) for r,img,order in zip(rows,images,orders)]
    batch=to_device(call_processor(texts,images))
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
        shown=last_logits(batch)[:,CHOICE_IDS].float().cpu().numpy()
    restored=np.stack([restore_choice_order(z,order) for z,order in zip(shown,orders)])
    return restored,softmax_numpy(restored,CFG["inference"]["temperature"])

def evaluate_in_memory(frame,shifts,image_policy=None):
    set_model_mode(False)
    logits=np.empty((len(shifts),len(frame),4),dtype=np.float32)
    probs=np.empty_like(logits)
    size=CFG["training"]["per_device_eval_batch_size"]
    cursor=0
    while cursor<len(frame):
        end=min(cursor+size,len(frame))
        rows=frame.iloc[cursor:end].to_dict("records")
        try:
            for k,shift in enumerate(shifts):
                z,p=forward_shift(rows,shift,image_policy)
                logits[k,cursor:end]=z
                probs[k,cursor:end]=p
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if size==1:
                raise
            size=max(1,size//2)
            continue
        cursor=end
    return logits,probs


## 18. High-Resolution 1024·4-way TTA·동일 모델 라우팅


기본 recipe는 native aspect+shift0/2. 1024 letterbox+4-way는 별도 validation 진단으로 남긴다.
라우팅은 낮은 confidence/margin 문항에서 같은 모델의 추가 shift만 호출한다.
별도 모델 앙상블이 아니며 threshold를 Public 점수로 조정하지 않는다.


In [ ]:
def route_same_model(frame,base_probs,shifts):
    setting=CFG["inference"]["same_model_routing"]
    if not setting["enabled"]:
        return base_probs,None
    values=np.sort(base_probs,axis=1)
    flags=np.flatnonzero((values[:,-1]<setting["confidence_threshold"]) |
                         ((values[:,-1]-values[:,-2])<setting["margin_threshold"]))
    extra=[s for s in setting["extra_shifts"] if s not in shifts]
    if not extra:
        raise ValueError("routing 추가 shift가 기본 shift와 겹칩니다.")
    out=base_probs.copy()
    if len(flags):
        z,p=evaluate_in_memory(frame.iloc[flags],extra)
        out[flags]=(base_probs[flags]*len(shifts)+p.sum(0))/(len(shifts)+len(extra))
    else:
        z=np.empty((len(extra),0,4),dtype=np.float32)
        p=z.copy()
    detail={"indices":flags,"ids":frame.iloc[flags].id.tolist(),"shifts":extra,"logits":z,"probs":p}
    return out,detail

def highres_diagnostic():
    policy=deepcopy(CFG["image"])
    policy["resize_mode"]="letterbox"
    policy["letterbox"].update(enabled=True,side=1024)
    shifts=[0,1,2,3]
    logits,probs=evaluate_in_memory(valid_df,shifts,policy)
    labels=valid_df.answer.map(CHOICES.index).to_numpy()
    score=float((probs.mean(0).argmax(1)==labels).mean())
    for name,array in [("logits",logits),("probs",probs)]:
        save_npy(RUN_DIR/"analysis"/f"valid_highres1024_shift_{name}.npy",array)
    save_json(RUN_DIR/"analysis"/"highres1024_tta4.json",{"accuracy":score,
        "shifts":shifts,"image_policy":policy,"ids":valid_df.id.tolist(),
        "comparison_scope":"same validation; separate diagnostic; not automatically applied to test"})
    return score


## 19. 학습 전 평가 → 2단계 학습 실행


여기서 모델의 학습 전 정확도를 기록한 후 두 단계를 실행한다.
Benchmark를 선택하면 측정 후 adapter·RNG를 복원한다. prepare에서는 실행하지 않는다.
infer는 지정 adapter를 사용하며 재학습하지 않는다.


In [ ]:
def execute_training():
    if RUN_MODE=="infer":
        return []
    before_path=RUN_DIR/"analysis"/"valid_before_probs.npy"
    if not before_path.exists():
        z,p=evaluate_in_memory(valid_df,CFG["inference"]["shifts"])
        save_npy(RUN_DIR/"analysis"/"valid_before_shift_logits.npy",z)
        save_npy(before_path,p.mean(0))
        save_json(RUN_DIR/"analysis"/"valid_before_ids.json",valid_df.id.tolist())
    else:
        if json.loads((RUN_DIR/"analysis"/"valid_before_ids.json").read_text())!=valid_df.id.tolist():
            raise ValueError("학습 전 평가 ID가 달라졌습니다.")
    stages=prepare_stages()
    if RUN_BENCHMARK:
        benchmark_training(stages[-1],CFG["training"]["benchmark_steps"])
    results=[train_one_stage(stage) for stage in stages]
    save_json(RUN_DIR/"training_summary.json",{"stages":results,"mode":RUN_MODE})
    return results

if GPU_RUN:
    training_results=execute_training()
    ACTIVE_ADAPTER_PATH=Path(ADAPTER_PATH) if ADAPTER_PATH else RUN_DIR/"adapter"
    ACTIVE_ADAPTER_HASH=adapter_fingerprint(ACTIVE_ADAPTER_PATH)
    print("현재 adapter:",ACTIVE_ADAPTER_PATH)


## 20. Test 추론·제출 ID/클래스 검증


추론/NPY 저장 함수는 다음 셀에 있다. 제출은 soft argmax로 생성하고 sample_submission ID와 연결한다.
smoke의 소량 CSV는 `submission_SMOKE_...`로 표시해 실제 제출과 구분한다.
누락 문항을 임의 답으로 채우지 않는다.


In [ ]:
def make_submission(frame,probs):
    check_probabilities(probs,len(frame))
    result=pd.DataFrame({"id":frame.id.tolist(),"answer":[CHOICES[i] for i in probs.argmax(1)]})
    sample=CFG["data"]["sample_submission_csv"]
    if sample and RUN_MODE!="smoke":
        template=pd.read_csv(sample,dtype=str,keep_default_na=False)
        if "id" not in template or template.id.duplicated().any() or set(template.id)!=set(result.id):
            raise ValueError("test와 sample_submission ID 집합이 다릅니다.")
        result=result.set_index("id").loc[template.id].reset_index()
    if result.id.duplicated().any() or not result.answer.isin(CHOICES).all():
        raise ValueError("제출 ID/답 검증 실패")
    prefix="submission_SMOKE_" if RUN_MODE=="smoke" else "submission_"
    path=RUN_DIR/(prefix+RUN_DIR.name+".csv")
    result.to_csv(path,index=False,encoding="utf-8")
    pd.DataFrame({"id":frame.id.tolist(),**{c:probs[:,j] for j,c in enumerate(CHOICES)}}).to_csv(
        RUN_DIR/"test_soft.csv",index=False,encoding="utf-8")
    return path


## 21. 실제 Logits·Probs·Labels·IDs·Resume 저장


요청한 기본 파일을 `probs/`에 남긴다. 실제 shift0 logits와 최종 TTA probs의 의미 차이를 metadata에 명시한다.
추가 shift 출력과 기준 shift probs도 저장한다. 매 batch checkpoint의 서명이 같은 경우만 이어서 추론한다.


In [ ]:
def predict_and_store(frame,split):
    shifts=CFG["inference"]["shifts"]
    if 0 not in shifts:
        raise ValueError("현재 기본 logits_reference_shift=0이 실제 추론 shifts에 있어야 합니다.")
    ids=frame.id.tolist()
    if len(set(ids))!=len(ids) or not len(ids):
        raise ValueError("추론 ID 중복/빈 데이터")
    folder=RUN_DIR/"probs"
    folder.mkdir(exist_ok=True)
    signature=digest({"config":CONFIG_SIGNATURE,"data":data_fingerprint,"adapter":ACTIVE_ADAPTER_HASH,
        "rows":frame[["id","_hash","question",*CHOICES]].to_dict("records"),"shifts":shifts})
    progress_path=folder/f"{split}_progress.json"
    raw_logits_path=folder/f"{split}_shift_logits.npy"
    raw_probs_path=folder/f"{split}_shift_probs.npy"
    shape=(len(shifts),len(frame),4)
    if progress_path.exists():
        state=json.loads(progress_path.read_text())
        if state["signature"]!=signature:
            raise ValueError(f"{split} 추론 재개 서명이 다릅니다.")
        cursor=state["completed_rows"]
        z=np.lib.format.open_memmap(raw_logits_path,mode="r+")
        p=np.lib.format.open_memmap(raw_probs_path,mode="r+")
        if z.shape!=shape or p.shape!=shape:
            raise ValueError("추론 재개 shape 오류")
    else:
        cursor=0
        z=np.lib.format.open_memmap(raw_logits_path,mode="w+",dtype="float32",shape=shape)
        p=np.lib.format.open_memmap(raw_probs_path,mode="w+",dtype="float32",shape=shape)
        z[:]=np.nan
        p[:]=np.nan
        z.flush();p.flush()
        save_json(progress_path,{"signature":signature,"completed_rows":0,"ids":ids})
    size=CFG["training"]["per_device_eval_batch_size"]
    set_model_mode(False)
    while cursor<len(frame):
        end=min(cursor+size,len(frame))
        rows=frame.iloc[cursor:end].to_dict("records")
        try:
            for k,shift in enumerate(shifts):
                zz,pp=forward_shift(rows,shift)
                z[k,cursor:end]=zz
                p[k,cursor:end]=pp
        except torch.cuda.OutOfMemoryError:
            gc.collect();torch.cuda.empty_cache()
            if size==1:
                save_json(RUN_DIR/"logs"/f"{split}_error.json",{"id":ids[cursor],"error":"OOM batch1"})
                raise
            size=max(1,size//2)
            continue
        except Exception as exc:
            save_json(RUN_DIR/"logs"/f"{split}_error.json",{"ids":ids[cursor:end],
                "type":type(exc).__name__,"error":str(exc),"completed_rows":cursor})
            raise
        z.flush();p.flush()
        cursor=end
        save_json(progress_path,{"signature":signature,"completed_rows":cursor,"ids":ids})
        if cursor%max(size*20,1)==0 or cursor==len(frame):
            print(split,cursor,"/",len(frame))
    if not np.isfinite(z).all() or not np.isfinite(p).all():
        raise ValueError("추론 logits/probs에 누락/NaN")
    final=np.asarray(p).mean(0,dtype=np.float32)
    final,routing=route_same_model(frame,final,shifts)
    check_probabilities(final,len(frame))
    ref=shifts.index(0)
    save_npy(folder/f"{split}_logits.npy",np.asarray(z[ref]))
    save_npy(folder/f"{split}_probs.npy",final)
    save_npy(folder/f"{split}_reference_probs.npy",np.asarray(p[ref]))
    save_json(folder/f"{split}_ids.json",ids)
    if routing is not None:
        save_npy(folder/f"{split}_routing_logits.npy",routing["logits"])
        save_npy(folder/f"{split}_routing_probs.npy",routing["probs"])
        save_json(folder/f"{split}_routing_ids.json",{"ids":routing["ids"],"indices":routing["indices"].tolist(),
                                                      "shifts":routing["shifts"]})
    meta_path=folder/"probability_metadata.json"
    meta=json.loads(meta_path.read_text()) if meta_path.exists() else {}
    meta[split]={"shape":list(final.shape),"shift_shape":list(shape),"class_order":CHOICES,
        "shifts":shifts,"temperature":CFG["inference"]["temperature"],"logits_reference_shift":0,
        "logits_semantics":"actual restored shift0 model logits",
        "probs_semantics":"conditional_same_model_shift_mean" if routing is not None else "arithmetic mean of restored shift probabilities",
        "reference_probs_semantics":"softmax of logits.npy at temperature",
        "hard_onehot_used":False,"signature":signature,"adapter_hash":ACTIVE_ADAPTER_HASH,
        "permutation_exempt_ids":[r["id"] for r in frame.to_dict("records") if not permutation_safe(r)],
        "is_smoke":RUN_MODE=="smoke","id_order":"validation manifest / original test order"}
    save_json(meta_path,meta)
    if split=="valid":
        labels=frame.answer.map(CHOICES.index).to_numpy(dtype=np.int64)
        save_npy(folder/"valid_labels.npy",labels)
        predictions=pd.DataFrame({"id":ids,"answer":frame.answer.tolist(),
            "pred":[CHOICES[j] for j in final.argmax(1)],"source":frame._src.tolist(),
            "permutation_safe":[permutation_safe(r) for r in frame.to_dict("records")],
            **{c:final[:,j] for j,c in enumerate(CHOICES)}})
        predictions["correct"]=predictions.pred.eq(predictions.answer)
        predictions.to_csv(RUN_DIR/"validation_predictions.csv",index=False,encoding="utf-8-sig")
        summary={"accuracy":float(predictions.correct.mean()),"n":len(frame),"shifts":shifts,
            "is_smoke":RUN_MODE=="smoke", "validation_signature":digest(frame[["id","answer","_hash","question",*CHOICES]].to_dict("records")),
            "by_class":predictions.groupby("answer").correct.agg(["count","mean"]).reset_index().to_dict("records"),
            "by_source":predictions.groupby("source").correct.agg(["count","mean"]).reset_index().to_dict("records"),
            "model":CFG["model"]["model_id"],"adapter_hash":ACTIVE_ADAPTER_HASH}
        before=RUN_DIR/"analysis"/"valid_before_probs.npy"
        if before.exists():
            prior=np.load(before,allow_pickle=False)
            if prior.shape!=final.shape:
                raise ValueError("학습 전/후 validation shape 다름")
            summary["before_accuracy"]=float((prior.argmax(1)==labels).mean())
        save_json(RUN_DIR/"validation_summary.json",summary)
    return np.asarray(z),np.asarray(p),final

def shift_analysis(frame,known_logits,known_probs):
    shifts=CFG["inference"]["shifts"]
    labels=frame.answer.map(CHOICES.index).to_numpy()
    records=[]
    all_logits=[]
    all_probs=[]
    candidates=CFG["inference"]["validation_shift_calibration"]["candidate_shifts"]
    for s in candidates:
        if s in shifts:
            z,p=known_logits[shifts.index(s)],known_probs[shifts.index(s)]
        else:
            zz,pp=evaluate_in_memory(frame,[s])
            z,p=zz[0],pp[0]
        all_logits.append(z)
        all_probs.append(p)
        records.append({"method":f"shift{s}","shifts":json.dumps([s]),"n":len(frame),
                        "accuracy":float((p.argmax(1)==labels).mean())})
    average=known_probs.mean(0)
    records.append({"method":"reference_tta","shifts":json.dumps(shifts),"n":len(frame),
                    "accuracy":float((average.argmax(1)==labels).mean())})
    pd.DataFrame(records).to_csv(RUN_DIR/"analysis"/"shift_calibration.csv",index=False)
    save_npy(RUN_DIR/"analysis"/"valid_calibration_shift_logits.npy",np.stack(all_logits))
    save_npy(RUN_DIR/"analysis"/"valid_calibration_shift_probs.npy",np.stack(all_probs))
    save_json(RUN_DIR/"analysis"/"selected_shift.json",{"selection":"reference_recipe",
        "selected_shifts":shifts,"candidate_shifts":candidates,"auto_search_applied":False,
        "reason":"초기 recipe 고정; 비교 수치는 진단용. Public 최적화로 선택하지 않음",
        "validation_ids":frame.id.tolist()})
    return records

if GPU_RUN:
    valid_shift_logits,valid_shift_probs,valid_probs=predict_and_store(valid_df,"valid")
    if CFG["inference"]["validation_shift_calibration"]["enabled"]:
        shift_records=shift_analysis(valid_df,valid_shift_logits,valid_shift_probs)
    else:
        pd.DataFrame([{"method":"reference_tta","shifts":json.dumps(CFG["inference"]["shifts"]),
            "accuracy":float((valid_probs.argmax(1)==valid_df.answer.map(CHOICES.index).to_numpy()).mean()),
            "n":len(valid_df)}]).to_csv(RUN_DIR/"analysis"/"shift_calibration.csv",index=False)
        save_json(RUN_DIR/"analysis"/"selected_shift.json",{"selected_shifts":CFG["inference"]["shifts"],"selection":"reference_recipe"})
    if CFG["inference"]["generation_diagnostic"]["enabled"]:
        generation_diagnostic(valid_df,CFG["inference"]["generation_diagnostic"]["sample_count"]).to_csv(
            RUN_DIR/"analysis"/"generation_diagnostic.csv",index=False,encoding="utf-8-sig")
    if RUN_HIGHRES_TTA4:
        print("1024/4-way validation:",highres_diagnostic())
    if test_df is not None:
        test_shift_logits,test_shift_probs,test_probs=predict_and_store(test_df,"test")
        submission_path=make_submission(test_df,test_probs)
        print("저장:",submission_path)
    print("validation:",json.loads((RUN_DIR/"validation_summary.json").read_text())["accuracy"])


## 22. 실제 결과 차트


실제 학습 로그와 예측으로만 loss/LR, 전후 정확도, 클래스/출처별 정확도, 혼동행렬, shift 비교를 그린다.
HTML 차트를 run에 저장한다. 목표 Public0.97과 참고 성적을 실측 차트로 넣지 않는다.


In [ ]:
def plot_results(folder):
    import plotly.express as px
    import plotly.graph_objects as go
    folder=Path(folder)
    charts=folder/"charts"
    charts.mkdir(exist_ok=True)
    figures=[]
    for path in sorted((folder/"logs").glob("*_history.csv")):
        history=pd.read_csv(path)
        for metric in ["loss","learning_rate","grad_norm"]:
            if metric in history and "step" in history:
                part=history.dropna(subset=[metric])
                if len(part):
                    figures.append((path.stem+"_"+metric,px.line(part,x="step",y=metric,title=path.stem+" "+metric)))
    summary=json.loads((folder/"validation_summary.json").read_text())
    rows=[{"stage":"after","accuracy":summary["accuracy"]}]
    if "before_accuracy" in summary:
        rows.insert(0,{"stage":"before","accuracy":summary["before_accuracy"]})
    figures.append(("before_after",px.bar(pd.DataFrame(rows),x="stage",y="accuracy",range_y=[0,1],title="동일 validation 학습 전후")))
    for name in ["by_class","by_source"]:
        frame=pd.DataFrame(summary[name])
        if len(frame):
            x="answer" if name=="by_class" else "source"
            figures.append((name,px.bar(frame,x=x,y="mean",hover_data=["count"],range_y=[0,1],title=name)))
    preds=pd.read_csv(folder/"validation_predictions.csv",dtype={"id":str})
    matrix=pd.crosstab(preds.answer,preds.pred).reindex(index=CHOICES,columns=CHOICES,fill_value=0)
    figures.append(("confusion_matrix",go.Figure(go.Heatmap(z=matrix.values,x=CHOICES,y=CHOICES))))
    shift_file=folder/"analysis"/"shift_calibration.csv"
    if shift_file.exists():
        shift=pd.read_csv(shift_file)
        figures.append(("shift_accuracy",px.bar(shift,x="method",y="accuracy",range_y=[0,1],title="동일 validation shift 진단")))
    training_file=folder/"training_summary.json"
    if training_file.exists():
        stages=pd.DataFrame(json.loads(training_file.read_text())["stages"])
        if len(stages):
            figures.append(("stage_vram",px.bar(stages,x="stage",y="peak_allocated_gib",title="실제 학습 peak GPU memory")))
    for name,figure in figures:
        figure.write_html(charts/(name+".html"),include_plotlyjs=True)
    print("실제 결과 차트 저장:",charts)
    return figures

if GPU_RUN:
    result_figures=plot_results(RUN_DIR)
    for _,figure in result_figures[:2]:
        figure.show()


## 23. 실행 결과 비교·Rescue/Harm


비교할 run 경로를 03의 COMPARE_RUN_DIRS에 넣는다. validation ID·정답·문항 hash가 같은 run만 비교한다.
데이터가 다른 과거 671/672/1343/1434 점수는 직접 순위 비교에서 제외한다.


In [ ]:
def compare_runs(paths):
    records=[]
    prediction_sets={}
    expected=None
    for folder in map(Path,paths):
        file=folder/"validation_summary.json"
        if not file.exists():
            records.append({"run":str(folder),"comparable":False,"reason":"summary 없음"})
            continue
        info=json.loads(file.read_text())
        sig=info["validation_signature"]
        if expected is None:
            expected=sig
        if sig!=expected:
            records.append({"run":str(folder),"comparable":False,"reason":"validation ID/정답/문항 hash 다름"})
            continue
        prediction_sets[str(folder)]=pd.read_csv(folder/"validation_predictions.csv",dtype={"id":str}).set_index("id")
        records.append({"run":str(folder),"comparable":True,"n":info["n"],"accuracy":info["accuracy"],
            "before_accuracy":info.get("before_accuracy"),"model":info["model"],"shifts":str(info["shifts"]),
            "adapter_hash":info["adapter_hash"],"is_smoke":info["is_smoke"]})
    if prediction_sets:
        anchor_name,anchor=next(iter(prediction_sets.items()))
        a=anchor.pred.eq(anchor.answer)
        for row in records:
            if row.get("comparable"):
                other=prediction_sets[row["run"]].loc[anchor.index]
                b=other.pred.eq(other.answer)
                row.update(anchor=anchor_name,rescue=int((~a&b).sum()),harm=int((a&~b).sum()),
                           net=int((~a&b).sum()-(a&~b).sum()))
    return pd.DataFrame(records)

if AUTO_RUN and COMPARE_RUN_DIRS:
    comparison=compare_runs(COMPARE_RUN_DIRS)
    comparison.to_csv(RUN_DIR/"analysis"/"run_comparison.csv",index=False,encoding="utf-8-sig")
    print(comparison.to_string(index=False))


## 24. 최종 감사·Top-2 검토 도구·Run 설명


run README에는 실제 수행한 범위와 결과/옵션을 적는다.
Top-2 밖 문항 선별 함수는 실제 확률+ID를 입력받아 사람 검토 CSV를 만들며 라벨을 바꾸지 않는다.
정확한 참고 교사 checkpoint가 미확인이므로 새 교사 모델을 자동으로 골라 호출하지 않는다.


In [ ]:
def export_top2_review(frame,ids,probs,path):
    if set(frame.id)!=set(map(str,ids)) or len(set(ids))!=len(ids):
        raise ValueError("교사 확률 ID와 정답 데이터가 다릅니다.")
    order={str(v):i for i,v in enumerate(ids)}
    p=np.asarray(probs)[[order[v] for v in frame.id]]
    check_probabilities(p,len(frame))
    gold=frame.answer.map(CHOICES.index).to_numpy()
    top2=np.argsort(p,axis=1)[:,-2:]
    outside=~(top2==gold[:,None]).any(1)
    review=frame.loc[outside,["id","path","question",*CHOICES,"answer"]].copy()
    review["teacher_top2"]=["/".join(CHOICES[j] for j in row[::-1]) for row in top2[outside]]
    review["review_status"]="pending_human_review"
    review.to_csv(path,index=False,encoding="utf-8-sig")
    return review

def write_run_readme():
    lines=["# VQA run",f"- mode: {RUN_MODE}",f"- model: {CFG['model']['model_id']}",
        f"- LoRA r/alpha: {CFG['lora']['r']}/{CFG['lora']['alpha']}",
        "- split: grouped 80:20, validation original order",
        f"- aux: {AUX_MODE}",f"- prompt repeat: {CFG['prompt']['repeat']}",
        f"- inference shifts: {CFG['inference']['shifts']}",
        "- logits.npy: actual restored reference shift0 logits",
        "- probs.npy: final restored probability mean; routing if enabled",
        "- softmax(logits.npy) corresponds to reference_probs.npy, not generally to probs.npy",
        "- reference Public0.97229: reported in A186 solution, not score of this run",
        "- Public/Private: unmeasured by this notebook", ""]
    summary=RUN_DIR/"validation_summary.json"
    if summary.exists():
        info=json.loads(summary.read_text())
        lines += [f"- measured validation: {info['accuracy']:.6f} ({info['n']} rows)",
                  f"- validation signature: {info['validation_signature']}",""]
    else:
        lines += ["- prepare only: no model training/inference/NPY generated",""]
    lines += ["## Files", "- config.json / environment.json / model_metadata.json",
        "- split_manifest.json / analysis/data_summary.json / cleaning_audit.csv",
        "- checkpoints / adapter_external / adapter",
        "- probs/valid_logits.npy, valid_probs.npy, valid_labels.npy, valid_ids.json",
        "- probs/test_logits.npy, test_probs.npy, test_ids.json",
        "- probs/*_shift_logits.npy, *_shift_probs.npy, *_reference_probs.npy",
        "- validation_predictions.csv / validation_summary.json",
        "- analysis/shift_calibration.csv / selected_shift.json",
        "- submission_<run_id>.csv; SMOKE file is partial and not for submission",
        "- charts/*.html / logs/*_history.csv", "",
        "## Limits", "- Source cleaning data and final score evidence were not in the supplied ZIP.",
        "- Exact-byte SHA grouping does not exclude every near duplicate; optional pHash needs review.",
        "- Legacy profiles preserve key settings; they do not reproduce every historical run.",
        "- This v1 uses the final checkpoint by default. Best-retained selection is only among retained checkpoints.",
        "- Validation tuning/selection performance is not an independent held-out result."]
    (RUN_DIR/"README.md").write_text("\n".join(lines)+"\n",encoding="utf-8")

if AUTO_RUN:
    write_run_readme()
    if GPU_RUN:
        required=[RUN_DIR/p for p in CFG["artifacts"]["required"] if test_df is not None or "test_" not in p]
        missing=[str(p) for p in required if not p.is_file()]
        if missing:
            raise RuntimeError("최종 필수 산출물 누락: "+str(missing))
        save_json(RUN_DIR/"execution_status.json",{"mode":RUN_MODE,"training_executed":RUN_MODE!="infer",
            "validation_inferred":True,"test_inferred":test_df is not None,"public_private_measured":False,
            "is_smoke":RUN_MODE=="smoke"})
    else:
        save_json(RUN_DIR/"execution_status.json",{"mode":RUN_MODE,"training_executed":False,
            "validation_inferred":False,"test_inferred":False,"public_private_measured":False})
    print("run 저장:",RUN_DIR)


### 출처·구현 차이·실행 전 확인

- A186 ZIP `src/train_gemma4_31b.py`: 모델/LoRA/프롬프트/답1토큰 CE/외부 변환/각1epoch/shift0,2.
- 기존 공통/hybrid/Gemma/Agnes/Qwen Only: 환경/로그/증강/보조정보/Direct Logit/shift/NPY/차트 기능 참고.
- 이 v1의 새 구현: grouped4:1, 균등 정답 위치, CPU prepare, 재개 서명, 실제 shift logits·확률 저장,
  원래 ID 순서·제출 ID 검증, 결과 비교. 고정 clean7169 manifest 재현과는 다르다.
- OCR/DINO의 범용 query·photometric 시작값은 설정에 공개했다. 기존 특정 성적의 정확한 재현값으로 주장하지 않는다.
- 임의 GPU에 적합성을 확정하지 않았다. 처음에는 smoke/backward/VRAM을 확인한다.
- Qwen GPTQ/Agnes는 모델별 라이브러리와 backend 호환성 검증이 특히 필요하다.
- `best_retained_checkpoint`는 남아 있는 checkpoint+final 중 validation 기준 선택이며 독립 평가가 아니다.
- 고해상도 진단은 자동으로 test에 반영하지 않는다. test까지 적용하려면 03의 image/shifts를 바꾸고 새 run으로 실행한다.
- 실제 대회 데이터/정제 CSV/GPU가 제공되지 않아 이 작성 세션에서 전체 모델 실험은 실행하지 않았다.

참고 링크:
[모델](https://huggingface.co/unsloth/gemma-4-31B-it-unsloth-bnb-4bit),
[MTVQA](https://huggingface.co/datasets/ByteDance/MTVQA),
[KRETA](https://huggingface.co/datasets/tabtoyou/KRETA),
[Grounding DINO API](https://huggingface.co/docs/transformers/model_doc/grounding-dino),
[StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).
